# CLSA retinal and epigenetic aging

## Quick use

Run this notebook after Algorithm Fairness notebook 01 has completed.
It reuses the complete quality-passing participant-visit RETFound vector
rollup, reads the six released baseline methylation phenotypes directly
from CoPv7_Qx_CANUE_PA_BS.csv, and writes a separate resumable analysis
under 08_epigenetics.

The notebook performs four related analyses:

1. Fit a participant-grouped, out-of-fold RETFound head for chronological
   age across the full eligible CLSA cohort and report standard regression
   metrics, calibration, Bland-Altman plots, and age-stratified accuracy.
2. Fit separate participant-grouped heads for the two absolute epigenetic
   ages (Horvath DNAmAge and Hannum age) among baseline participants with
   each measurement. These are secondary prediction analyses.
3. Compare chronological age, out-of-fold retinal age from the
   chronological-age head, and each absolute epigenetic age, including
   three-dimensional plots and participant-paired distance tests.
4. Quantify whether released racial/cultural background, spirometry
   ethnicity, or sex explains variation in the age gaps using adjusted
   nested regression and incremental R-squared.

Important guardrails:

- Epigenetic measurements are baseline phenotypes, so three-way analyses
  use baseline fundus vectors only.
- DNAmAge_COM and Hannum_Age_COM are ages in years. The released
  acceleration difference, residual, IEAA, and EEAA variables remain on
  their released acceleration scales and are never treated as ages.
- The primary retinal-versus-epigenetic comparison uses retinal age
  predicted by a chronological-age model that never used epigenetic
  outcomes. Agreement from an epigenetic-trained retinal head is reported
  separately because using it to prove shared biology would be circular.
- All model assessment is out of fold, all splitting is by participant,
  and all inferential rows are participant-level.
- Positive retinal-age gap means older-appearing retina; negative means
  younger-appearing retina. A gap is a relative biomarker, not a diagnosis.

In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib
import json
import math
import os
import re
import shutil
import sys
import uuid
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    brier_score_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold,
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(
    style="whitegrid",
    context="notebook",
    font_scale=1.08,
    rc={
        "axes.titleweight": "semibold",
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "figure.titlesize": 15,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    },
)
pd.set_option("display.max_columns", 100)

In [ ]:
repo_root = Path(
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina"
)
dataset_root = Path(
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset"
)
derived_root = dataset_root / "derived" / "clsa_retinal_aging"
fairness_root = derived_root / "Age_Glaucoma" / "16_algorithm_fairness"

participant_embedding_path = (
    fairness_root / "01_private" / "participant_visit_embeddings.parquet"
)
race_path = fairness_root / "01_private" / "racial_background_private.parquet"
sap_questionnaire_path = derived_root / "sap_questionnaire_visit"
baseline_archive_path = (
    dataset_root / "2209017_UOttawa_EFreeman_BL.zip"
)
baseline_member_suffix = "CoPv7_Qx_CANUE_PA_BS.csv"

output_root = fairness_root / "08_epigenetics"
private_root = output_root / "01_private"
model_root = output_root / "02_models"
statistics_root = output_root / "03_statistics"
figure_root = output_root / "04_figures"
checkpoint_root = output_root / "05_checkpoints"
for directory in (
    private_root,
    model_root,
    statistics_root,
    figure_root,
    checkpoint_root,
):
    directory.mkdir(parents=True, exist_ok=True)

expected_embedding_dim = 1024
ridge_alpha = 10.0
cv_folds = 5
bootstrap_repetitions = 2000
permutation_repetitions = 10000
minimum_model_participants = 50
minimum_reporting_group_n = 10
minimum_age_bin_n = 30
baseline_chunk_size = 100_000
maximum_scatter_points = 5000
prediction_bootstrap_repetitions = 500
minimum_binary_outcome_events = 30
acceleration_extreme_quantile = 0.80
questionnaire_row_chunk_size = 500
minimum_questionnaire_observed = 100
minimum_questionnaire_level_n = 10
maximum_questionnaire_categorical_levels = 20
random_state = 20260903
resume_completed_outputs = True

module_root = repo_root / "src"
if not module_root.exists():
    raise FileNotFoundError(f"Repository source directory missing: {module_root}")
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

import fundus_retfound_pipeline as _fundus
import retfound_fairness as _fairness
_fundus = importlib.reload(_fundus)
_fairness = importlib.reload(_fairness)
from fundus_retfound_pipeline import (
    AgeModelConfig,
    train_age_head,
    write_frame,
    write_json,
)
from retfound_fairness import benjamini_hochberg

print("Output root:", output_root)
print("Retinal vectors:", participant_embedding_path)

## 1. Reusable statistical and checkpoint helpers

Confidence intervals and permutation tests resample participants. The
training checkpoint signature includes participant, visit, outcome, image
count, and the consolidated embedding source metadata.

In [ ]:
def normalize_identifier(series):
    return series.astype("string").str.strip()


def normalize_visit(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace({"FUP1": "F1"})
    )


def normalized_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )


def harmonize_sex(series):
    return normalized_code(series).map(
        {"F": "Female", "2": "Female", "M": "Male", "1": "Male"}
    )


def label_spirometry_ethnicity(series):
    labels = {
        "1": "Caucasian",
        "2": "Asian",
        "3": "African",
        "4": "Hispanic",
        "5": "Other ethnicity",
    }
    codes = normalized_code(series)
    missing_codes = {
        "",
        "-77771",
        "-77772",
        "-88880",
        "-88888",
        "-99991",
        "-99993",
        "-99999",
    }
    released_text = (
        series.astype("string").str.strip().replace("", pd.NA)
    )
    released_text = released_text.mask(codes.isin(missing_codes))
    return codes.map(labels).fillna(released_text)


def stable_value(series):
    values = series.dropna()
    if values.empty:
        return None
    modes = values.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else values.iloc[0]


def source_signature(path):
    path = Path(path)
    stat = path.stat()
    return f"{path}|{stat.st_size}|{stat.st_mtime_ns}"


def frame_signature(frame, target_column, source_token):
    rows = (
        frame[["participant_id", "visit", target_column, "n_embedded_images"]]
        .astype(str)
        .agg("|".join, axis=1)
        .sort_values()
    )
    digest = hashlib.sha256()
    digest.update(source_token.encode("utf-8"))
    for value in rows:
        digest.update(value.encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def publish_file(local_path, destination_path):
    local_path = Path(local_path)
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    partial = destination_path.with_name(
        f".{destination_path.name}.{uuid.uuid4().hex}.partial"
    )
    try:
        shutil.copy2(local_path, partial)
        if partial.stat().st_size != local_path.stat().st_size:
            raise OSError("Published file size mismatch")
        os.replace(partial, destination_path)
    finally:
        partial.unlink(missing_ok=True)
    return destination_path


def assign_group_folds(frame, max_splits):
    groups = frame["participant_id"].astype(str).to_numpy()
    n_splits = min(max_splits, len(np.unique(groups)))
    fold = np.zeros(len(frame), dtype=int)
    splitter = GroupKFold(n_splits=n_splits)
    dummy = np.zeros((len(frame), 1), dtype=float)
    target = frame["age"].to_numpy(float)
    for fold_number, (_, test_index) in enumerate(
        splitter.split(dummy, target, groups), 1
    ):
        fold[test_index] = fold_number
    if not np.all(fold > 0):
        raise RuntimeError("Some training rows were not assigned an OOF fold")
    return fold


def fit_or_resume_head(frame, target_column, model_name):
    work = frame[
        [
            "participant_id",
            "visit",
            target_column,
            "embedding",
            "n_embedded_images",
        ]
    ].copy()
    work.attrs = {}
    work[target_column] = pd.to_numeric(work[target_column], errors="coerce")
    work = work.dropna(
        subset=["participant_id", "visit", target_column, "embedding"]
    ).reset_index(drop=True)
    if work["participant_id"].nunique() < minimum_model_participants:
        raise ValueError(
            f"{model_name} has only {work['participant_id'].nunique()} "
            f"participants; at least {minimum_model_participants} required"
        )
    work = work.rename(columns={target_column: "age"})
    signature = frame_signature(
        work,
        "age",
        (
            source_signature(participant_embedding_path)
            + f"|{model_name}|alpha={ridge_alpha}|folds={cv_folds}"
            + f"|seed={random_state}|calibration=intercept"
        ),
    )
    destination = model_root / model_name
    destination.mkdir(parents=True, exist_ok=True)
    model_path = destination / f"{model_name}.joblib"
    oof_path = destination / f"{model_name}_oof.parquet"
    metadata_path = destination / f"{model_name}_metadata.json"
    can_resume = False
    if (
        resume_completed_outputs
        and model_path.is_file()
        and oof_path.is_file()
        and metadata_path.is_file()
    ):
        metadata = json.loads(metadata_path.read_text())
        can_resume = metadata.get("training_signature") == signature
    if can_resume:
        oof = pd.read_parquet(oof_path)
        bundle = joblib.load(model_path)
        print(f"Resumed {model_name}: {len(oof):,} OOF rows")
        return oof, bundle

    local_root = (
        Path("/local_disk0/tmp")
        / f"epigenetics-{model_name}-{uuid.uuid4().hex}"
    )
    local_root.mkdir(parents=True, exist_ok=True)
    oof, bundle = train_age_head(
        work,
        local_root,
        AgeModelConfig(
            alpha=ridge_alpha,
            max_splits=cv_folds,
            calibration="intercept",
            random_state=random_state,
        ),
        write_metadata=False,
    )
    oof.attrs = {}
    oof["cv_fold"] = assign_group_folds(work, cv_folds)
    oof["target_name"] = target_column
    bundle.update(
        {
            "model_name": model_name,
            "target_name": target_column,
            "training_signature": signature,
        }
    )
    local_model = local_root / f"{model_name}.joblib"
    local_oof = local_root / f"{model_name}_oof.parquet"
    joblib.dump(bundle, local_model)
    write_frame(oof, local_oof)
    publish_file(local_model, model_path)
    publish_file(local_oof, oof_path)
    write_json(
        {
            "training_signature": signature,
            "target_name": target_column,
            "n_participants": int(work["participant_id"].nunique()),
            "n_participant_visits": int(len(work)),
            "n_images_represented": int(work["n_embedded_images"].sum()),
            "ridge_alpha": ridge_alpha,
            "cv_folds": int(oof["cv_splits"].iloc[0]),
            "grouped_by_participant": True,
            "model_path": str(model_path),
            "oof_path": str(oof_path),
        },
        metadata_path,
    )
    shutil.rmtree(local_root, ignore_errors=True)
    print(f"Fitted {model_name}: {len(oof):,} OOF rows")
    return oof, bundle


def concordance_correlation(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    covariance = np.cov(x, y, ddof=1)[0, 1]
    denominator = (
        np.var(x, ddof=1)
        + np.var(y, ddof=1)
        + (np.mean(x) - np.mean(y)) ** 2
    )
    return float(2 * covariance / denominator) if denominator > 0 else np.nan


def regression_metrics(frame, target_column, prediction_column, label):
    work = frame[[target_column, prediction_column]].apply(
        pd.to_numeric, errors="coerce"
    ).dropna()
    target = work[target_column].to_numpy(float)
    prediction = work[prediction_column].to_numpy(float)
    error = prediction - target
    if len(work) < 3:
        raise ValueError(f"Insufficient observations for {label}")
    slope, intercept = np.polyfit(target, prediction, 1)
    pearson = stats.pearsonr(target, prediction)
    spearman = stats.spearmanr(target, prediction)
    return {
        "analysis": label,
        "n": int(len(work)),
        "target_mean": float(np.mean(target)),
        "target_sd": float(np.std(target, ddof=1)),
        "prediction_mean": float(np.mean(prediction)),
        "prediction_sd": float(np.std(prediction, ddof=1)),
        "mae": float(mean_absolute_error(target, prediction)),
        "median_absolute_error": float(np.median(np.abs(error))),
        "rmse": float(np.sqrt(mean_squared_error(target, prediction))),
        "r2": float(r2_score(target, prediction)),
        "pearson_r": float(pearson.statistic),
        "pearson_p": float(pearson.pvalue),
        "spearman_rho": float(spearman.statistic),
        "spearman_p": float(spearman.pvalue),
        "calibration_slope": float(slope),
        "calibration_intercept": float(intercept),
        "mean_error": float(np.mean(error)),
        "sd_error": float(np.std(error, ddof=1)),
        "ccc": concordance_correlation(target, prediction),
    }


def bootstrap_mean_ci(values, repetitions, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    estimates = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        estimates[index] = np.mean(
            values[rng.integers(0, len(values), len(values))]
        )
    return tuple(np.quantile(estimates, [0.025, 0.975]))


def signflip_less_test(differences, repetitions, seed):
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) < 2:
        return np.nan
    observed = float(np.mean(differences))
    rng = np.random.default_rng(seed)
    exceed = 1
    for _ in range(repetitions):
        signs = rng.choice([-1.0, 1.0], size=len(differences))
        exceed += float(np.mean(differences * signs)) <= observed
    return float(exceed / (repetitions + 1))


def coerce_binary(series):
    codes = normalized_code(series)
    mapped = codes.map(
        {
            "1": 1.0,
            "Y": 1.0,
            "YES": 1.0,
            "TRUE": 1.0,
            "T": 1.0,
            "0": 0.0,
            "2": 0.0,
            "N": 0.0,
            "NO": 0.0,
            "FALSE": 0.0,
            "F": 0.0,
        }
    )
    numeric = pd.to_numeric(series, errors="coerce")
    mapped = mapped.fillna(
        numeric.where(numeric.isin([0, 1]))
    )
    return mapped.astype(float)


def safe_zscore(series):
    values = pd.to_numeric(series, errors="coerce")
    standard_deviation = float(values.std(ddof=1))
    if not np.isfinite(standard_deviation) or standard_deviation <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float)
    return (values - float(values.mean())) / standard_deviation


def cross_fitted_spline_residual(
    frame,
    estimate_column,
    age_column,
    group_column,
    n_splits=cv_folds,
):
    work = frame[
        [estimate_column, age_column, group_column]
    ].copy()
    work[estimate_column] = pd.to_numeric(
        work[estimate_column], errors="coerce"
    )
    work[age_column] = pd.to_numeric(
        work[age_column], errors="coerce"
    )
    valid = work[
        [estimate_column, age_column, group_column]
    ].notna().all(axis=1)
    expected = pd.Series(np.nan, index=frame.index, dtype=float)
    valid_work = work.loc[valid]
    groups = valid_work[group_column].astype(str).to_numpy()
    splits = min(n_splits, len(np.unique(groups)))
    if splits < 2:
        raise ValueError(
            f"Cannot cross-fit {estimate_column}: fewer than two groups"
        )
    splitter = GroupKFold(n_splits=splits)
    x = valid_work[[age_column]].to_numpy(float)
    y = valid_work[estimate_column].to_numpy(float)
    for train_index, test_index in splitter.split(x, y, groups):
        calibration = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=5,
                        degree=3,
                        include_bias=False,
                    ),
                ),
                ("ridge", Ridge(alpha=1.0)),
            ]
        )
        calibration.fit(x[train_index], y[train_index])
        expected.loc[
            valid_work.index[test_index]
        ] = calibration.predict(x[test_index])
    residual = (
        pd.to_numeric(frame[estimate_column], errors="coerce")
        - expected
    )
    return residual, expected


def bootstrap_metric_difference(
    outcome,
    full_prediction,
    base_prediction,
    metric,
    repetitions,
    seed,
):
    outcome = np.asarray(outcome, dtype=float)
    full_prediction = np.asarray(full_prediction, dtype=float)
    base_prediction = np.asarray(base_prediction, dtype=float)
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(repetitions):
        sample = rng.integers(0, len(outcome), len(outcome))
        sampled_outcome = outcome[sample]
        if np.unique(sampled_outcome).size < 2:
            continue
        differences.append(
            metric(
                sampled_outcome,
                full_prediction[sample],
            )
            - metric(
                sampled_outcome,
                base_prediction[sample],
            )
        )
    if not differences:
        return np.nan, np.nan
    return tuple(np.quantile(differences, [0.025, 0.975]))

## 2. Load complete participant-visit vectors and questionnaire covariates

The consolidated input was produced from every completed RETFound embedding
batch after technical-quality filtering in notebook 01. No images or
vectors are recalculated here.

In [ ]:
required_paths = {
    "participant-visit embedding rollup": participant_embedding_path,
    "racial-background table": race_path,
    "baseline questionnaire ZIP": baseline_archive_path,
    "SAP questionnaire Delta table": sap_questionnaire_path,
}
missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]
if missing_paths:
    raise FileNotFoundError(
        "Required completed inputs are missing:\n- "
        + "\n- ".join(missing_paths)
    )

embeddings = pd.read_parquet(participant_embedding_path)
embeddings.attrs = {}
required_embedding_columns = {
    "participant_id",
    "visit",
    "embedding",
    "n_embedded_images",
}
missing = required_embedding_columns - set(embeddings.columns)
if missing:
    raise ValueError(f"Embedding rollup missing columns: {sorted(missing)}")
embeddings["participant_id"] = normalize_identifier(
    embeddings["participant_id"]
)
embeddings["visit"] = normalize_visit(embeddings["visit"])
if embeddings.duplicated(["participant_id", "visit"]).any():
    raise ValueError("Embedding rollup is not unique by participant and visit")
dimensions = embeddings["embedding"].map(
    lambda value: np.asarray(value).reshape(-1).size
)
if set(dimensions) != {expected_embedding_dim}:
    raise ValueError(
        f"Unexpected embedding dimensions: {sorted(set(dimensions))}"
    )

candidate_sap_columns = [
    "participant_id",
    "visit",
    "age_at_fundus_years",
    "sex_at_birth",
    "ethnicity_spirometry",
    "diabetes",
    "hypertension",
    "heart_disease",
    "stroke",
    "kidney_disease",
    "chronic_kidney_disease",
    "arthritis_any",
    "osteoporosis",
    "asthma_or_copd",
    "cancer",
    "low_back_pain",
    "depression_cesd10",
    "smoking_status",
    "bmi",
    "body_mass_index",
    "physical_activity",
    "self_rated_health",
    "multimorbidity_selected_count",
    "education_level_sap_harmonized",
    "household_income_band",
]
sap_spark = spark.read.format("delta").load(str(sap_questionnaire_path))
required_sap = {"participant_id", "visit", "age_at_fundus_years"}
missing = required_sap - set(sap_spark.columns)
if missing:
    raise ValueError(f"SAP questionnaire table missing: {sorted(missing)}")
available_sap_columns = [
    column for column in candidate_sap_columns
    if column in sap_spark.columns
]
sap = sap_spark.select(*available_sap_columns).toPandas()
sap.attrs = {}
sap["participant_id"] = normalize_identifier(sap["participant_id"])
sap["visit"] = normalize_visit(sap["visit"])
sap["age_at_fundus_years"] = pd.to_numeric(
    sap["age_at_fundus_years"], errors="coerce"
)
aggregations = {"age_at_fundus_years": "median"}
for column in available_sap_columns:
    if column not in {"participant_id", "visit", "age_at_fundus_years"}:
        aggregations[column] = stable_value
sap = (
    sap.groupby(["participant_id", "visit"], as_index=False)
    .agg(aggregations)
)
if sap.duplicated(["participant_id", "visit"]).any():
    raise ValueError("Collapsed SAP table remains non-unique")
if "sex_at_birth" in sap:
    sap["sex_labeled"] = harmonize_sex(sap["sex_at_birth"])
if "ethnicity_spirometry" in sap:
    sap["ethnicity_spirometry_labeled"] = label_spirometry_ethnicity(
        sap["ethnicity_spirometry"]
    )

race = pd.read_parquet(race_path)
race.attrs = {}
race["participant_id"] = normalize_identifier(race["participant_id"])
race = race.drop_duplicates("participant_id", keep="last")
race_columns = [
    column for column in (
        "participant_id",
        "racial_background",
        "racial_background_detail",
        "racial_background_status",
    )
    if column in race.columns
]
race = race[race_columns]

participant_visit = (
    embeddings.merge(
        sap,
        on=["participant_id", "visit"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        race,
        on="participant_id",
        how="left",
        validate="many_to_one",
    )
)
participant_visit["chronological_age"] = pd.to_numeric(
    participant_visit["age_at_fundus_years"], errors="coerce"
)
write_frame(
    participant_visit.drop(columns=["embedding"]),
    private_root / "participant_visit_covariates_private.parquet",
)
print(
    f"Loaded {len(participant_visit):,} participant-visits from "
    f"{participant_visit['participant_id'].nunique():,} participants, "
    f"representing {int(participant_visit['n_embedded_images'].sum()):,} "
    "quality-passing images"
)

## 3. Read every released baseline epigenetic phenotype

The large CSV is streamed in chunks and the matched six-column result is
checkpointed. Any participant with at least one released measurement is
retained. Missing-value sentinels are converted to missing without
inventing additional epigenetic variables or QC fields.

In [ ]:
EPIGENETIC_SOURCE_VARIABLES = {
    "DNAmAge_COM": "epigenetic_dnam_age",
    "AgeAccelerationDifference_COM": (
        "epigenetic_age_acceleration_difference"
    ),
    "AgeAccelerationResidual_COM": (
        "epigenetic_age_acceleration_residual"
    ),
    "IEAA_COM": "epigenetic_ieaa",
    "EEAA_COM": "epigenetic_eeaa",
    "Hannum_Age_COM": "epigenetic_hannum_age",
}
ABSOLUTE_CLOCKS = {
    "epigenetic_dnam_age": "Horvath DNAm age",
    "epigenetic_hannum_age": "Hannum epigenetic age",
}
EPIGENETIC_MODEL_NAMES = {
    "epigenetic_dnam_age": "retfound_epigenetic_dnam_age",
    "epigenetic_hannum_age": "retfound_epigenetic_hannum_age",
}
ACCELERATION_MEASURES = {
    "epigenetic_age_acceleration_difference": (
        "Released age-acceleration difference"
    ),
    "epigenetic_age_acceleration_residual": (
        "Released age-acceleration residual"
    ),
    "epigenetic_ieaa": "IEAA",
    "epigenetic_eeaa": "EEAA",
}
numeric_missing_codes = {
    "",
    "-8",
    "-77771",
    "-77772",
    "-88880",
    "-88888",
    "-99991",
    "-99993",
    "-99999",
}

with zipfile.ZipFile(baseline_archive_path) as archive:
    member_matches = [
        name for name in archive.namelist()
        if name.endswith(baseline_member_suffix)
    ]
    if len(member_matches) != 1:
        raise ValueError(
            f"Expected one member ending {baseline_member_suffix!r}; "
            f"found {len(member_matches)}"
        )
    baseline_member = member_matches[0]
    with archive.open(baseline_member) as stream:
        baseline_header = pd.read_csv(stream, nrows=0).columns.tolist()

id_candidates = [
    "entity_id",
    "participant_id",
    "ID",
    "id",
    "Entity_ID",
    "ENTITY_ID",
]
baseline_id_column = next(
    (column for column in id_candidates if column in baseline_header),
    None,
)
if baseline_id_column is None:
    raise ValueError("Unable to identify baseline participant ID column")
required_raw = {
    baseline_id_column,
    *EPIGENETIC_SOURCE_VARIABLES,
}
missing = required_raw - set(baseline_header)
if missing:
    raise ValueError(
        "Baseline questionnaire CSV is missing supplied epigenetic "
        f"columns: {sorted(missing)}"
    )

epigenetic_checkpoint = (
    checkpoint_root / "baseline_epigenetic_phenotypes_private.parquet"
)
epigenetic_metadata_path = (
    checkpoint_root / "baseline_epigenetic_phenotypes_metadata.json"
)
epigenetic_signature = hashlib.sha256(
    (
        source_signature(baseline_archive_path)
        + "|"
        + baseline_member
        + "|"
        + "|".join(sorted(EPIGENETIC_SOURCE_VARIABLES))
    ).encode("utf-8")
).hexdigest()
resume_epigenetic = False
if (
    resume_completed_outputs
    and epigenetic_checkpoint.is_file()
    and epigenetic_metadata_path.is_file()
):
    metadata = json.loads(epigenetic_metadata_path.read_text())
    resume_epigenetic = (
        metadata.get("source_signature") == epigenetic_signature
    )

if resume_epigenetic:
    epigenetic = pd.read_parquet(epigenetic_checkpoint)
    print("Resumed baseline epigenetic checkpoint")
else:
    eligible_ids = set(
        participant_visit["participant_id"].dropna().astype(str)
    )
    retained_chunks = []
    scanned_rows = 0
    with zipfile.ZipFile(baseline_archive_path) as archive:
        with archive.open(baseline_member) as stream:
            for chunk_number, chunk in enumerate(
                pd.read_csv(
                    stream,
                    usecols=[
                        baseline_id_column,
                        *EPIGENETIC_SOURCE_VARIABLES,
                    ],
                    dtype="string",
                    chunksize=baseline_chunk_size,
                    low_memory=False,
                ),
                1,
            ):
                scanned_rows += len(chunk)
                chunk[baseline_id_column] = normalize_identifier(
                    chunk[baseline_id_column]
                )
                retained = chunk[
                    chunk[baseline_id_column].isin(eligible_ids)
                ].copy()
                if not retained.empty:
                    retained_chunks.append(retained)
                print(
                    f"[epigenetic {chunk_number}] scanned "
                    f"{scanned_rows:,}; retained {len(retained):,}",
                    flush=True,
                )
    raw = (
        pd.concat(retained_chunks, ignore_index=True)
        if retained_chunks
        else pd.DataFrame(
            columns=[
                baseline_id_column,
                *EPIGENETIC_SOURCE_VARIABLES,
            ]
        )
    )
    raw = raw.rename(columns={baseline_id_column: "participant_id"})
    raw["participant_id"] = normalize_identifier(raw["participant_id"])
    if raw["participant_id"].duplicated().any():
        raise ValueError(
            "Baseline epigenetic source is not unique by participant"
        )
    epigenetic = raw[["participant_id"]].copy()
    for source_column, analysis_column in (
        EPIGENETIC_SOURCE_VARIABLES.items()
    ):
        values = raw[source_column].astype("string").str.strip()
        values = values.mask(values.isin(numeric_missing_codes))
        epigenetic[analysis_column] = pd.to_numeric(
            values, errors="coerce"
        )
    epigenetic["any_epigenetic_measure"] = epigenetic[
        list(EPIGENETIC_SOURCE_VARIABLES.values())
    ].notna().any(axis=1)
    epigenetic = epigenetic[
        epigenetic["any_epigenetic_measure"]
    ].reset_index(drop=True)
    write_frame(epigenetic, epigenetic_checkpoint)
    write_json(
        {
            "source_signature": epigenetic_signature,
            "baseline_member": baseline_member,
            "n_participants_any_measure": int(len(epigenetic)),
            "columns": list(EPIGENETIC_SOURCE_VARIABLES.values()),
        },
        epigenetic_metadata_path,
    )

epigenetic_columns = list(EPIGENETIC_SOURCE_VARIABLES.values())
coverage = pd.DataFrame(
    [
        {
            "measure": column,
            "label": {
                **ABSOLUTE_CLOCKS,
                **ACCELERATION_MEASURES,
            }[column],
            "participants": int(epigenetic[column].notna().sum()),
        }
        for column in epigenetic_columns
    ]
)
coverage.loc[len(coverage)] = {
    "measure": "any_epigenetic_measure",
    "label": "Any released epigenetic phenotype",
    "participants": int(
        epigenetic["any_epigenetic_measure"].sum()
    ),
}
write_frame(coverage, statistics_root / "epigenetic_coverage.csv")
display(coverage)

## 4. Fit or resume participant-grouped out-of-fold retinal heads

The chronological head uses every participant-visit with age. Each
epigenetic head uses every baseline participant with that absolute clock.
Both eyes and repeated images have already been averaged within
participant-visit, preventing image-rich participants from dominating.

In [ ]:
chronological_training = participant_visit.dropna(
    subset=["chronological_age", "embedding"]
).copy()
chronological_oof, chronological_bundle = fit_or_resume_head(
    chronological_training,
    "chronological_age",
    "retfound_chronological_age",
)

baseline_vectors = (
    participant_visit[participant_visit["visit"].eq("BL")]
    .merge(
        epigenetic,
        on="participant_id",
        how="inner",
        validate="one_to_one",
    )
)
if baseline_vectors["participant_id"].duplicated().any():
    raise ValueError(
        "Baseline epigenetic model frame is not participant-unique"
    )

epigenetic_oof = {}
epigenetic_bundles = {}
for clock_column in ABSOLUTE_CLOCKS:
    available = baseline_vectors.dropna(
        subset=[clock_column, "embedding"]
    ).copy()
    model_name = EPIGENETIC_MODEL_NAMES[clock_column]
    oof, bundle = fit_or_resume_head(
        available,
        clock_column,
        model_name,
    )
    epigenetic_oof[clock_column] = oof
    epigenetic_bundles[clock_column] = bundle

print(
    "Chronological model participants:",
    chronological_oof["participant_id"].nunique(),
)
for clock_column, oof in epigenetic_oof.items():
    print(
        ABSOLUTE_CLOCKS[clock_column],
        "model participants:",
        oof["participant_id"].nunique(),
    )

## 5. Standard model metrics and age-distribution accuracy

Primary chronological performance is one record per participant, choosing
baseline before follow-up. Participant-visit results are retained as a
sensitivity analysis. The central 80% and two tails are prespecified from
chronological-age deciles; fixed five-year bins show where calibration or
MAE deteriorates without selecting a favorable range after seeing results.

In [ ]:
chronological_oof["participant_id"] = normalize_identifier(
    chronological_oof["participant_id"]
)
chronological_oof["visit"] = normalize_visit(
    chronological_oof["visit"]
)
chronological_oof = chronological_oof.rename(
    columns={
        "age": "chronological_age",
        "retinal_age_prediction_oof": "retinal_age_oof",
        "retinal_age_raw_oof": "retinal_age_raw_oof",
    }
)
chronological_oof["retinal_chronological_gap"] = (
    chronological_oof["retinal_age_oof"]
    - chronological_oof["chronological_age"]
)
visit_priority = {"BL": 0, "F1": 1}
chronological_oof["_visit_priority"] = (
    chronological_oof["visit"].map(visit_priority).fillna(99)
)
chronological_participant = (
    chronological_oof.sort_values(
        ["participant_id", "_visit_priority"],
        kind="stable",
    )
    .drop_duplicates("participant_id", keep="first")
    .drop(columns="_visit_priority")
    .reset_index(drop=True)
)

metric_rows = [
    regression_metrics(
        chronological_oof,
        "chronological_age",
        "retinal_age_oof",
        "Chronological age head: participant-visit",
    ),
    regression_metrics(
        chronological_participant,
        "chronological_age",
        "retinal_age_oof",
        "Chronological age head: participant",
    ),
]
for clock_column, oof in epigenetic_oof.items():
    metric_rows.append(
        regression_metrics(
            oof,
            "age",
            "retinal_age_prediction_oof",
            f"{ABSOLUTE_CLOCKS[clock_column]} head: baseline participant",
        )
    )
model_metrics = pd.DataFrame(metric_rows)
write_frame(model_metrics, statistics_root / "model_standard_metrics.csv")
display(model_metrics.round(4))

age_values = chronological_participant["chronological_age"].to_numpy(float)
age_q10, age_q90 = np.quantile(age_values, [0.10, 0.90])
chronological_participant["age_distribution_stratum"] = np.select(
    [
        chronological_participant["chronological_age"] < age_q10,
        chronological_participant["chronological_age"] > age_q90,
    ],
    ["Lower 10% age tail", "Upper 10% age tail"],
    default="Central 80%",
)
lower_edge = math.floor(np.nanmin(age_values) / 5) * 5
upper_edge = math.ceil(np.nanmax(age_values) / 5) * 5 + 5
age_edges = np.arange(lower_edge, upper_edge + 0.1, 5)
chronological_participant["age_bin_5y"] = pd.cut(
    chronological_participant["chronological_age"],
    bins=age_edges,
    right=False,
    include_lowest=True,
)

def grouped_performance(frame, group_column, analysis):
    rows = []
    for group, subset in frame.groupby(group_column, observed=True):
        if len(subset) < minimum_age_bin_n:
            continue
        row = regression_metrics(
            subset,
            "chronological_age",
            "retinal_age_oof",
            analysis,
        )
        row[group_column] = str(group)
        rows.append(row)
    return pd.DataFrame(rows)

tail_performance = grouped_performance(
    chronological_participant,
    "age_distribution_stratum",
    "Chronological head by age-distribution stratum",
)
age_bin_performance = grouped_performance(
    chronological_participant,
    "age_bin_5y",
    "Chronological head by five-year age bin",
)
overall_mae = float(
    model_metrics.loc[
        model_metrics["analysis"].eq(
            "Chronological age head: participant"
        ),
        "mae",
    ].iloc[0]
)
if not age_bin_performance.empty:
    age_bin_performance["exploratory_accuracy_flag"] = (
        age_bin_performance["mae"].le(overall_mae)
        & age_bin_performance["mean_error"].abs().le(1.0)
    )
write_frame(
    tail_performance,
    statistics_root / "chronological_model_tail_performance.csv",
)
write_frame(
    age_bin_performance,
    statistics_root / "chronological_model_5y_age_bin_performance.csv",
)
display(tail_performance.round(4))
display(age_bin_performance.round(4))

In [ ]:
performance_panels = [
    (
        "Chronological age",
        chronological_participant,
        "chronological_age",
        "retinal_age_oof",
        model_metrics.iloc[1],
    )
]
for clock_column, oof in epigenetic_oof.items():
    metric = model_metrics[
        model_metrics["analysis"].str.startswith(
            ABSOLUTE_CLOCKS[clock_column]
        )
    ].iloc[0]
    performance_panels.append(
        (
            ABSOLUTE_CLOCKS[clock_column],
            oof,
            "age",
            "retinal_age_prediction_oof",
            metric,
        )
    )

figure, axes = plt.subplots(
    len(performance_panels),
    2,
    figsize=(14, 4.7 * len(performance_panels)),
    squeeze=False,
    layout="constrained",
)
for row_index, (
    label,
    frame,
    target_column,
    prediction_column,
    metric,
) in enumerate(performance_panels):
    plot = frame[[target_column, prediction_column]].apply(
        pd.to_numeric, errors="coerce"
    ).dropna()
    if len(plot) > maximum_scatter_points:
        plot = plot.sample(
            maximum_scatter_points,
            random_state=random_state + row_index,
        )
    target = plot[target_column].to_numpy(float)
    prediction = plot[prediction_column].to_numpy(float)
    limits = [
        min(target.min(), prediction.min()),
        max(target.max(), prediction.max()),
    ]
    axes[row_index, 0].hexbin(
        target,
        prediction,
        gridsize=45,
        mincnt=1,
        cmap="viridis",
    )
    axes[row_index, 0].plot(limits, limits, "--", color="black")
    axes[row_index, 0].set(
        title=(
            f"RETFound prediction of {label}\n"
            f"MAE={metric['mae']:.2f} y; "
            f"R²={metric['r2']:.3f}; r={metric['pearson_r']:.3f}"
        ),
        xlabel=f"Observed {label} (years)",
        ylabel=f"OOF predicted {label} (years)",
    )
    mean_pair = (target + prediction) / 2
    difference = prediction - target
    mean_difference = float(np.mean(difference))
    loa = 1.96 * float(np.std(difference, ddof=1))
    axes[row_index, 1].scatter(
        mean_pair,
        difference,
        s=12,
        alpha=0.35,
    )
    axes[row_index, 1].axhline(
        mean_difference,
        color="black",
        linewidth=1.5,
    )
    axes[row_index, 1].axhline(
        mean_difference + loa,
        color="gray",
        linestyle="--",
    )
    axes[row_index, 1].axhline(
        mean_difference - loa,
        color="gray",
        linestyle="--",
    )
    axes[row_index, 1].set(
        title=f"Bland-Altman: {label}",
        xlabel="Mean observed and predicted age (years)",
        ylabel="Predicted minus observed (years)",
    )
model_performance_figure = (
    figure_root / "figure_1_oof_model_performance.png"
)
figure.savefig(
    model_performance_figure,
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.2,
)
plt.show()
plt.close(figure)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.3),
    layout="constrained",
)
if not age_bin_performance.empty:
    x = np.arange(len(age_bin_performance))
    age_bin_labels = [
        str(value)
        .replace("[", "")
        .replace(")", "")
        .replace(", ", "–<")
        for value in age_bin_performance["age_bin_5y"]
    ]
    axes[0].plot(
        x,
        age_bin_performance["mae"],
        marker="o",
        label="MAE",
    )
    axes[0].plot(
        x,
        age_bin_performance["rmse"],
        marker="s",
        label="RMSE",
    )
    axes[0].axhline(
        overall_mae,
        color="black",
        linestyle="--",
        label="Overall MAE",
    )
    axes[0].set_xticks(
        x,
        age_bin_labels,
        rotation=35,
        ha="right",
    )
    axes[0].set(
        title="Chronological-age error by five-year age bin",
        xlabel="Chronological age",
        ylabel="Error (years)",
    )
    axes[0].legend()
    axes[1].plot(
        x,
        age_bin_performance["mean_error"],
        marker="o",
        color="#C44E52",
    )
    axes[1].axhline(0, color="black", linestyle="--")
    axes[1].fill_between(
        [-0.5, len(x) - 0.5],
        [-1, -1],
        [1, 1],
        color="gray",
        alpha=0.12,
    )
    axes[1].set_xticks(
        x,
        age_bin_labels,
        rotation=35,
        ha="right",
    )
    axes[1].set(
        title="Calibration bias by five-year age bin",
        xlabel="Chronological age",
        ylabel="Mean OOF retinal-age gap (years)",
    )
figure.savefig(
    figure_root / "figure_2_chronological_age_operating_range.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.2,
)
plt.show()
plt.close(figure)

## 6. Assemble the baseline three-age cohort

Retinal age comes only from the chronological-age head's held-out
prediction. The two epigenetic-trained head predictions are attached under
explicitly secondary names.

In [ ]:
retinal_baseline = chronological_oof[
    chronological_oof["visit"].eq("BL")
][
    [
        "participant_id",
        "chronological_age",
        "retinal_age_oof",
        "retinal_chronological_gap",
        "cv_fold",
        "n_embedded_images",
    ]
].copy()
master = (
    retinal_baseline.merge(
        epigenetic,
        on="participant_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        participant_visit[
            participant_visit["visit"].eq("BL")
        ].drop(columns=["embedding", "chronological_age"], errors="ignore"),
        on=["participant_id"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_questionnaire"),
    )
)
for clock_column, oof in epigenetic_oof.items():
    secondary = oof[
        ["participant_id", "retinal_age_prediction_oof"]
    ].rename(
        columns={
            "retinal_age_prediction_oof": (
                f"retfound_prediction_of_{clock_column}_oof"
            )
        }
    )
    master = master.merge(
        secondary,
        on="participant_id",
        how="left",
        validate="one_to_one",
    )

for clock_column in ABSOLUTE_CLOCKS:
    short = (
        "dnam"
        if clock_column == "epigenetic_dnam_age"
        else "hannum"
    )
    master[f"{short}_chronological_gap"] = (
        master[clock_column] - master["chronological_age"]
    )
    master[f"retinal_{short}_gap"] = (
        master["retinal_age_oof"] - master[clock_column]
    )
    master[f"absolute_retinal_{short}_distance"] = (
        master[f"retinal_{short}_gap"].abs()
    )
    master[f"absolute_chronological_{short}_distance"] = (
        master[f"{short}_chronological_gap"].abs()
    )
master["absolute_retinal_chronological_distance"] = (
    master["retinal_chronological_gap"].abs()
)

# Residual retinal age acceleration uses the full participant-level OOF
# cohort to estimate the expected age trend, then applies it to the
# epigenetic subset.
valid_retinal = chronological_participant[
    ["retinal_age_oof", "chronological_age"]
].dropna()
retinal_slope, retinal_intercept = np.polyfit(
    valid_retinal["chronological_age"],
    valid_retinal["retinal_age_oof"],
    1,
)
master["retinal_age_acceleration_residual"] = (
    master["retinal_age_oof"]
    - (
        retinal_intercept
        + retinal_slope * master["chronological_age"]
    )
)
write_frame(
    master,
    private_root / "baseline_three_age_master_private.parquet",
)
print(
    f"Baseline participants with any epigenetic phenotype and OOF retinal "
    f"age: {len(master):,}"
)

## 7. Chronological age versus released epigenetic ages

Correlation alone is insufficient for agreement, so the table also reports
MAE, RMSE, signed bias, calibration, and concordance correlation.

In [ ]:
agreement_rows = []
for clock_column, clock_label in ABSOLUTE_CLOCKS.items():
    subset = master.dropna(
        subset=[
            "chronological_age",
            "retinal_age_oof",
            clock_column,
        ]
    )
    agreement_rows.extend(
        [
            regression_metrics(
                subset,
                "chronological_age",
                clock_column,
                f"{clock_label} versus chronological age",
            ),
            regression_metrics(
                subset,
                "chronological_age",
                "retinal_age_oof",
                f"Retinal age versus chronological age: {clock_label} subset",
            ),
            regression_metrics(
                subset,
                clock_column,
                "retinal_age_oof",
                f"Retinal age versus {clock_label}",
            ),
        ]
    )
agreement = pd.DataFrame(agreement_rows)
write_frame(
    agreement,
    statistics_root / "three_age_agreement_metrics.csv",
)
display(agreement.round(4))

figure, axes = plt.subplots(
    2,
    len(ABSOLUTE_CLOCKS),
    figsize=(7.2 * len(ABSOLUTE_CLOCKS), 9.5),
    squeeze=False,
    layout="constrained",
)
for column_index, (clock_column, clock_label) in enumerate(
    ABSOLUTE_CLOCKS.items()
):
    subset = master.dropna(
        subset=["chronological_age", clock_column]
    ).copy()
    x = subset["chronological_age"].to_numpy(float)
    y = subset[clock_column].to_numpy(float)
    limits = [min(x.min(), y.min()), max(x.max(), y.max())]
    axes[0, column_index].hexbin(
        x,
        y,
        gridsize=40,
        mincnt=1,
        cmap="mako",
    )
    axes[0, column_index].plot(limits, limits, "--", color="black")
    axes[0, column_index].set(
        title=(
            f"{clock_label} versus chronological age\n"
            f"n={len(subset):,}"
        ),
        xlabel="Chronological age (years)",
        ylabel=f"{clock_label} (years)",
    )
    difference = y - x
    axes[1, column_index].scatter(
        (x + y) / 2,
        difference,
        s=13,
        alpha=0.4,
    )
    mean_difference = np.mean(difference)
    loa = 1.96 * np.std(difference, ddof=1)
    axes[1, column_index].axhline(
        mean_difference,
        color="black",
    )
    axes[1, column_index].axhline(
        mean_difference + loa,
        color="gray",
        linestyle="--",
    )
    axes[1, column_index].axhline(
        mean_difference - loa,
        color="gray",
        linestyle="--",
    )
    axes[1, column_index].set(
        title=f"Bland-Altman: {clock_label}",
        xlabel="Mean of chronological and epigenetic age (years)",
        ylabel="Epigenetic − chronological age (years)",
    )
figure.savefig(
    figure_root / "figure_3_epigenetic_vs_chronological_age.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.2,
)
plt.show()
plt.close(figure)

## 8. Demographic stratification and explanatory contribution

Released racial/cultural background, spirometry ethnicity, and sex are
evaluated separately. Categories below the minimum reporting count are
excluded from group-specific inference. Nested models first include
chronological age (linear and quadratic), available cardiometabolic
comorbidities, and the complementary demographic covariate; the full model
then adds the demographic factor. Incremental R-squared quantifies how much
additional variance the demographic factor explains.

In [ ]:
demographic_variables = {
    "racial_background": "Self-reported racial/cultural background",
    "ethnicity_spirometry_labeled": "Spirometry ethnicity",
    "sex_labeled": "Sex at birth",
}
gap_variables = {
    "retinal_chronological_gap": "Retinal minus chronological age",
    "dnam_chronological_gap": "Horvath minus chronological age",
    "hannum_chronological_gap": "Hannum minus chronological age",
    "retinal_dnam_gap": "Retinal minus Horvath age",
    "retinal_hannum_gap": "Retinal minus Hannum age",
    "retinal_age_acceleration_residual": (
        "Age-residualized retinal age acceleration"
    ),
}

comorbidity_covariates = [
    column
    for column in (
        "diabetes",
        "hypertension",
        "heart_disease",
        "stroke",
        "multimorbidity_selected_count",
    )
    if column in master.columns
]
for column in comorbidity_covariates:
    master[column] = pd.to_numeric(master[column], errors="coerce")

# Model performance stratification is reported separately from the
# adjusted age-gap models below.
participant_demographics = participant_visit[
    [
        column
        for column in (
            "participant_id",
            "visit",
            *demographic_variables,
        )
        if column in participant_visit.columns
    ]
].copy()
participant_demographics["_visit_priority"] = (
    participant_demographics["visit"]
    .map({"BL": 0, "F1": 1})
    .fillna(99)
)
participant_demographics = (
    participant_demographics.sort_values(
        ["participant_id", "_visit_priority"],
        kind="stable",
    )
    .drop_duplicates("participant_id")
    .drop(columns=["visit", "_visit_priority"])
)
head_frames = [
    (
        "Chronological age head",
        chronological_participant.rename(
            columns={
                "chronological_age": "target_age",
                "retinal_age_oof": "prediction_age",
            }
        ),
    )
]
for clock_column, oof in epigenetic_oof.items():
    head_frames.append(
        (
            f"{ABSOLUTE_CLOCKS[clock_column]} head",
            oof.rename(
                columns={
                    "age": "target_age",
                    "retinal_age_prediction_oof": "prediction_age",
                }
            ),
        )
    )
stratified_performance_rows = []
for head_label, head_frame in head_frames:
    head_frame = head_frame.merge(
        participant_demographics,
        on="participant_id",
        how="left",
        validate="many_to_one",
    )
    for demographic, demographic_label in (
        demographic_variables.items()
    ):
        if demographic not in head_frame.columns:
            continue
        counts = head_frame[demographic].value_counts(dropna=True)
        for level, count in counts.items():
            if count < minimum_reporting_group_n:
                continue
            subset = head_frame[
                head_frame[demographic].eq(level)
            ]
            metrics = regression_metrics(
                subset,
                "target_age",
                "prediction_age",
                head_label,
            )
            metrics.update(
                {
                    "demographic": demographic,
                    "demographic_label": demographic_label,
                    "level": str(level),
                }
            )
            stratified_performance_rows.append(metrics)
stratified_head_performance = pd.DataFrame(
    stratified_performance_rows
)
write_frame(
    stratified_head_performance,
    statistics_root
    / "retfound_heads_demographic_performance.csv",
)

demographic_descriptive_rows = []
demographic_model_rows = []
demographic_coefficient_rows = []
for demographic_index, (
    demographic,
    demographic_label,
) in enumerate(demographic_variables.items()):
    if demographic not in master.columns:
        continue
    counts = master[demographic].value_counts(dropna=True)
    eligible_levels = counts[
        counts >= minimum_reporting_group_n
    ].index.astype(str)
    for outcome_index, (outcome, outcome_label) in enumerate(
        gap_variables.items()
    ):
        if outcome not in master.columns:
            continue
        work = master[
            master[demographic].astype("string").isin(eligible_levels)
        ].copy()
        work[outcome] = pd.to_numeric(work[outcome], errors="coerce")
        work = work.dropna(subset=[demographic, outcome])
        if work.empty or work[demographic].nunique() < 2:
            continue
        for level, subset in work.groupby(demographic):
            values = subset[outcome].to_numpy(float)
            low, high = bootstrap_mean_ci(
                values,
                bootstrap_repetitions,
                random_state
                + 100 * demographic_index
                + 10 * outcome_index,
            )
            demographic_descriptive_rows.append(
                {
                    "demographic": demographic,
                    "demographic_label": demographic_label,
                    "level": str(level),
                    "outcome": outcome,
                    "outcome_label": outcome_label,
                    "n": int(len(values)),
                    "mean": float(np.mean(values)),
                    "sd": float(np.std(values, ddof=1)),
                    "median": float(np.median(values)),
                    "ci_low": low,
                    "ci_high": high,
                }
            )

        work = work.copy()
        work["chronological_age_sq"] = (
            work["chronological_age"] ** 2
        )
        base_terms = [
            "chronological_age",
            "chronological_age_sq",
            *comorbidity_covariates,
        ]
        if demographic != "sex_labeled" and "sex_labeled" in work:
            base_terms.append("C(sex_labeled)")
        if (
            demographic == "sex_labeled"
            and "racial_background" in work
            and work["racial_background"].value_counts().ge(
                minimum_reporting_group_n
            ).sum() >= 2
        ):
            frequent_race = work[
                "racial_background"
            ].value_counts()
            frequent_race = frequent_race[
                frequent_race >= minimum_reporting_group_n
            ].index
            work = work[
                work["racial_background"].isin(frequent_race)
            ].copy()
            base_terms.append("C(racial_background)")
        model_columns = [
            outcome,
            demographic,
            "chronological_age",
            "chronological_age_sq",
            *comorbidity_covariates,
        ]
        if "C(sex_labeled)" in base_terms:
            model_columns.append("sex_labeled")
        if "C(racial_background)" in base_terms:
            model_columns.append("racial_background")
        model_work = work[
            list(dict.fromkeys(model_columns))
        ].dropna()
        if (
            len(model_work) < 30
            or model_work[demographic].nunique() < 2
        ):
            continue
        # statsmodels/formulaic can reject pandas StringDtype even
        # though the values are valid labels. Convert every formula
        # categorical to an explicit pandas category first.
        formula_categoricals = [demographic]
        if "C(sex_labeled)" in base_terms:
            formula_categoricals.append("sex_labeled")
        if "C(racial_background)" in base_terms:
            formula_categoricals.append("racial_background")
        for categorical in dict.fromkeys(formula_categoricals):
            model_work[categorical] = pd.Categorical(
                model_work[categorical].astype(str)
            )
        base_formula = (
            f"{outcome} ~ " + " + ".join(base_terms)
        )
        full_formula = (
            base_formula + f" + C({demographic})"
        )
        try:
            base_fit = smf.ols(
                base_formula,
                data=model_work,
            ).fit()
            full_fit = smf.ols(
                full_formula,
                data=model_work,
            ).fit(cov_type="HC3")
            classical_full = smf.ols(
                full_formula,
                data=model_work,
            ).fit()
            nested_f, nested_p, df_difference = (
                classical_full.compare_f_test(base_fit)
            )
            demographic_model_rows.append(
                {
                    "demographic": demographic,
                    "demographic_label": demographic_label,
                    "outcome": outcome,
                    "outcome_label": outcome_label,
                    "n": int(len(model_work)),
                    "n_levels": int(
                        model_work[demographic].nunique()
                    ),
                    "base_r2": float(base_fit.rsquared),
                    "full_r2": float(full_fit.rsquared),
                    "incremental_r2": float(
                        full_fit.rsquared - base_fit.rsquared
                    ),
                    "nested_f_statistic": float(nested_f),
                    "nested_f_p_value": float(nested_p),
                    "df_difference": float(df_difference),
                    "covariance": "HC3 for coefficients; classical nested F",
                    "status": "ok",
                }
            )
            confidence = full_fit.conf_int()
            term_prefix = f"C({demographic})[T."
            for term in full_fit.params.index:
                if not term.startswith(term_prefix):
                    continue
                demographic_coefficient_rows.append(
                    {
                        "demographic": demographic,
                        "demographic_label": demographic_label,
                        "outcome": outcome,
                        "outcome_label": outcome_label,
                        "term": term,
                        "comparison_level": (
                            term[len(term_prefix):-1]
                        ),
                        "n": int(len(model_work)),
                        "adjusted_coefficient": float(
                            full_fit.params[term]
                        ),
                        "ci_low": float(confidence.loc[term, 0]),
                        "ci_high": float(confidence.loc[term, 1]),
                        "p_value": float(full_fit.pvalues[term]),
                        "covariance": "HC3",
                    }
                )
        except Exception as error:
            demographic_model_rows.append(
                {
                    "demographic": demographic,
                    "demographic_label": demographic_label,
                    "outcome": outcome,
                    "outcome_label": outcome_label,
                    "n": int(len(model_work)),
                    "status": f"failed:{type(error).__name__}",
                    "error_message": str(error)[:500],
                }
            )

demographic_descriptives = pd.DataFrame(
    demographic_descriptive_rows
)
demographic_models = pd.DataFrame(demographic_model_rows)
demographic_coefficients = pd.DataFrame(
    demographic_coefficient_rows
)
if not demographic_models.empty:
    if "nested_f_p_value" not in demographic_models.columns:
        demographic_models["nested_f_p_value"] = np.nan
    if "incremental_r2" not in demographic_models.columns:
        demographic_models["incremental_r2"] = np.nan
    valid_p = pd.to_numeric(
        demographic_models["nested_f_p_value"],
        errors="coerce",
    )
    demographic_models["fdr_q_value"] = benjamini_hochberg(
        valid_p.to_numpy(float)
    )
    demographic_models["significant_fdr_0_05"] = (
        demographic_models["fdr_q_value"] < 0.05
    )
if not demographic_coefficients.empty:
    demographic_coefficients["fdr_q_value"] = benjamini_hochberg(
        demographic_coefficients["p_value"].to_numpy(float)
    )
    demographic_coefficients["significant_fdr_0_05"] = (
        demographic_coefficients["fdr_q_value"] < 0.05
    )
write_frame(
    demographic_descriptives,
    statistics_root / "demographic_gap_descriptives.csv",
)
write_frame(
    demographic_models,
    statistics_root / "demographic_incremental_models.csv",
)
write_frame(
    demographic_coefficients,
    statistics_root / "demographic_adjusted_coefficients.csv",
)
demographic_failures = demographic_models[
    ~demographic_models["status"].eq("ok")
].copy()
write_frame(
    demographic_failures,
    statistics_root / "demographic_model_failures.csv",
)
if not demographic_failures.empty:
    display(demographic_failures)
    first_failure = demographic_failures.iloc[0]
    raise RuntimeError(
        "Adjusted demographic models failed; the notebook will not "
        "report 'no demographic differences' from an incomplete run. "
        f"First failure: {first_failure['demographic_label']} / "
        f"{first_failure['outcome_label']}: "
        f"{first_failure.get('error_message', first_failure['status'])}. "
        "See demographic_model_failures.csv."
    )
display(
    demographic_models.sort_values(
        ["fdr_q_value", "incremental_r2"],
        ascending=[True, False],
    ).round(5)
)

selected_outcomes = list(gap_variables)
plot_data = demographic_descriptives[
    demographic_descriptives["outcome"].isin(selected_outcomes)
].copy()
if not plot_data.empty:
    for demographic, demographic_label in (
        demographic_variables.items()
    ):
        figure, axes = plt.subplots(
            3,
            2,
            figsize=(14, 13),
            squeeze=False,
            layout="constrained",
        )
        for axis, outcome in zip(
            axes.ravel(),
            selected_outcomes,
        ):
            subset = plot_data[
                plot_data["demographic"].eq(demographic)
                & plot_data["outcome"].eq(outcome)
            ].sort_values("mean")
            if subset.empty:
                axis.set_visible(False)
                continue
            point = subset["mean"].to_numpy(float)
            low = subset["ci_low"].to_numpy(float)
            high = subset["ci_high"].to_numpy(float)
            xerr = np.vstack(
                [
                    np.maximum(
                        point - np.minimum(low, point),
                        0.0,
                    ),
                    np.maximum(
                        np.maximum(high, point) - point,
                        0.0,
                    ),
                ]
            )
            position = np.arange(len(subset))
            axis.errorbar(
                point,
                position,
                xerr=xerr,
                fmt="o",
                capsize=3,
            )
            labels = [
                f"{level} (n={int(n):,})"
                for level, n in zip(
                    subset["level"],
                    subset["n"],
                )
            ]
            axis.set_yticks(position, labels)
            axis.axvline(0, color="black", linestyle="--")
            axis.set(
                title=(
                    gap_variables[outcome]
                ),
                xlabel="Mean age gap (years)",
                ylabel="",
            )
        figure.suptitle(
            f"Age-gap stratification: {demographic_label}",
            y=1.02,
        )
        figure.savefig(
            figure_root
            / f"figure_4_{demographic}_age_gaps.png",
            dpi=220,
            bbox_inches="tight",
            pad_inches=0.25,
        )
        plt.show()
        plt.close(figure)

model_plot = demographic_models[
    demographic_models["status"].eq("ok")
].copy()
if not model_plot.empty:
    model_plot["label"] = (
        model_plot["demographic_label"]
        + "\n"
        + model_plot["outcome_label"]
    )
    model_plot = model_plot.sort_values(
        "incremental_r2",
        ascending=True,
    )
    figure, axis = plt.subplots(
        figsize=(13, max(7, 0.55 * len(model_plot))),
        layout="constrained",
    )
    colors = np.where(
        model_plot["significant_fdr_0_05"],
        "#C44E52",
        "#4C72B0",
    )
    axis.barh(
        np.arange(len(model_plot)),
        model_plot["incremental_r2"],
        color=colors,
    )
    axis.set_yticks(
        np.arange(len(model_plot)),
        model_plot["label"],
    )
    axis.set(
        title=(
            "Additional age-gap variance explained by demographic factor"
        ),
        xlabel="Incremental R² beyond age and available covariates",
        ylabel="",
    )
    axis.text(
        0.99,
        0.01,
        "Red: omnibus FDR q<0.05",
        transform=axis.transAxes,
        ha="right",
        va="bottom",
        fontsize=10,
    )
    figure.savefig(
        figure_root
        / "figure_4d_demographic_incremental_r2.png",
        dpi=220,
        bbox_inches="tight",
        pad_inches=0.25,
    )
    plt.show()
    plt.close(figure)

## 9. Three-way paired distance tests

For each absolute epigenetic clock, the prespecified hypotheses are:

- absolute retinal-epigenetic distance is smaller than absolute
  retinal-chronological distance; and
- absolute retinal-epigenetic distance is smaller than absolute
  epigenetic-chronological distance.

Both one-sided Wilcoxon signed-rank and participant-level sign-flip tests
are reported, with FDR correction. These tests establish relative
closeness—not interchangeability, causality, or a shared biological
mechanism.

In [ ]:
distance_rows = []
distance_summary_rows = []
for clock_index, (clock_column, clock_label) in enumerate(
    ABSOLUTE_CLOCKS.items()
):
    short = (
        "dnam"
        if clock_column == "epigenetic_dnam_age"
        else "hannum"
    )
    subset = master.dropna(
        subset=[
            "retinal_age_oof",
            "chronological_age",
            clock_column,
        ]
    ).copy()
    distances = {
        "Retinal–epigenetic": np.abs(
            subset["retinal_age_oof"] - subset[clock_column]
        ).to_numpy(float),
        "Retinal–chronological": np.abs(
            subset["retinal_age_oof"]
            - subset["chronological_age"]
        ).to_numpy(float),
        "Epigenetic–chronological": np.abs(
            subset[clock_column]
            - subset["chronological_age"]
        ).to_numpy(float),
    }
    for pair_index, (pair_label, values) in enumerate(
        distances.items()
    ):
        low, high = bootstrap_mean_ci(
            values,
            bootstrap_repetitions,
            random_state + 1000 * clock_index + pair_index,
        )
        distance_summary_rows.append(
            {
                "clock": clock_column,
                "clock_label": clock_label,
                "pair": pair_label,
                "n": int(len(values)),
                "mean_absolute_distance": float(np.mean(values)),
                "median_absolute_distance": float(np.median(values)),
                "ci_low": low,
                "ci_high": high,
            }
        )

    retinal_epigenetic = distances["Retinal–epigenetic"]
    for comparison_index, comparator in enumerate(
        (
            "Retinal–chronological",
            "Epigenetic–chronological",
        )
    ):
        difference = (
            retinal_epigenetic - distances[comparator]
        )
        low, high = bootstrap_mean_ci(
            difference,
            bootstrap_repetitions,
            random_state
            + 5000
            + 100 * clock_index
            + comparison_index,
        )
        try:
            wilcoxon_p = float(
                stats.wilcoxon(
                    retinal_epigenetic,
                    distances[comparator],
                    alternative="less",
                    zero_method="wilcox",
                ).pvalue
            )
        except ValueError:
            wilcoxon_p = np.nan
        distance_rows.append(
            {
                "clock": clock_column,
                "clock_label": clock_label,
                "hypothesis": (
                    "Retinal–epigenetic absolute distance < "
                    f"{comparator} absolute distance"
                ),
                "n": int(len(difference)),
                "mean_distance_difference": float(
                    np.mean(difference)
                ),
                "mean_difference_ci_low": low,
                "mean_difference_ci_high": high,
                "proportion_retinal_epigenetic_smaller": float(
                    np.mean(difference < 0)
                ),
                "wilcoxon_one_sided_p": wilcoxon_p,
                "signflip_one_sided_p": signflip_less_test(
                    difference,
                    permutation_repetitions,
                    random_state
                    + 7000
                    + 100 * clock_index
                    + comparison_index,
                ),
            }
        )

distance_tests = pd.DataFrame(distance_rows)
distance_summary = pd.DataFrame(distance_summary_rows)
distance_tests["wilcoxon_fdr_q"] = benjamini_hochberg(
    distance_tests["wilcoxon_one_sided_p"].to_numpy(float)
)
distance_tests["signflip_fdr_q"] = benjamini_hochberg(
    distance_tests["signflip_one_sided_p"].to_numpy(float)
)
distance_tests["supports_closer_after_fdr"] = (
    distance_tests["mean_difference_ci_high"].lt(0)
    & distance_tests["wilcoxon_fdr_q"].lt(0.05)
    & distance_tests["signflip_fdr_q"].lt(0.05)
)
write_frame(
    distance_tests,
    statistics_root / "three_age_paired_distance_tests.csv",
)
write_frame(
    distance_summary,
    statistics_root / "three_age_distance_summary.csv",
)
display(distance_tests.round(5))

acceleration_rows = []
for measure, label in ACCELERATION_MEASURES.items():
    subset = master[
        ["retinal_age_acceleration_residual", measure]
    ].apply(pd.to_numeric, errors="coerce").dropna()
    if len(subset) < minimum_reporting_group_n:
        continue
    pearson = stats.pearsonr(
        subset["retinal_age_acceleration_residual"],
        subset[measure],
    )
    spearman = stats.spearmanr(
        subset["retinal_age_acceleration_residual"],
        subset[measure],
    )
    acceleration_rows.append(
        {
            "measure": measure,
            "label": label,
            "n": int(len(subset)),
            "pearson_r": float(pearson.statistic),
            "pearson_p": float(pearson.pvalue),
            "spearman_rho": float(spearman.statistic),
            "spearman_p": float(spearman.pvalue),
        }
    )
acceleration_correlations = pd.DataFrame(acceleration_rows)
if not acceleration_correlations.empty:
    acceleration_correlations["pearson_fdr_q"] = (
        benjamini_hochberg(
            acceleration_correlations["pearson_p"].to_numpy(float)
        )
    )
    acceleration_correlations["spearman_fdr_q"] = (
        benjamini_hochberg(
            acceleration_correlations["spearman_p"].to_numpy(float)
        )
    )
write_frame(
    acceleration_correlations,
    statistics_root / "retinal_epigenetic_acceleration_correlations.csv",
)
display(acceleration_correlations.round(5))

In [ ]:
figure = plt.figure(figsize=(11.5, 15))
three_age_plot_data = {}
for index, (clock_column, clock_label) in enumerate(
    ABSOLUTE_CLOCKS.items(),
    1,
):
    axis = figure.add_subplot(
        len(ABSOLUTE_CLOCKS),
        1,
        index,
        projection="3d",
    )
    subset = master.dropna(
        subset=[
            "chronological_age",
            "retinal_age_oof",
            clock_column,
        ]
    ).copy()
    if len(subset) > maximum_scatter_points:
        subset = subset.sample(
            maximum_scatter_points,
            random_state=random_state + index,
        )
    three_age_plot_data[clock_column] = subset
    color = np.abs(
        subset["retinal_age_oof"] - subset[clock_column]
    )
    scatter = axis.scatter(
        subset["chronological_age"],
        subset["retinal_age_oof"],
        subset[clock_column],
        c=color,
        cmap="magma",
        s=16,
        alpha=0.65,
    )
    lower = float(
        subset[
            [
                "chronological_age",
                "retinal_age_oof",
                clock_column,
            ]
        ].min().min()
    )
    upper = float(
        subset[
            [
                "chronological_age",
                "retinal_age_oof",
                clock_column,
            ]
        ].max().max()
    )
    diagonal = np.linspace(lower, upper, 100)
    axis.plot(
        diagonal,
        diagonal,
        diagonal,
        "--",
        color="black",
        linewidth=1,
    )
    axis.set_xlim(lower, upper)
    axis.set_ylim(lower, upper)
    axis.set_zlim(lower, upper)
    axis.set_box_aspect((1.15, 1.15, 1.0))
    axis.set(
        title=f"{clock_label} (n={len(subset):,})",
        xlabel="Chronological age (years)",
        ylabel="Retinal age (years)",
        zlabel=clock_label,
    )
    axis.tick_params(axis="both", which="major", labelsize=9)
    axis.xaxis.labelpad = 8
    axis.yaxis.labelpad = 8
    axis.zaxis.labelpad = 8
    axis.view_init(elev=24, azim=-55)
    figure.colorbar(
        scatter,
        ax=axis,
        shrink=0.55,
        pad=0.08,
        label="|Retinal − epigenetic| years",
    )
figure.suptitle(
    "Participant-level chronological, retinal, and epigenetic ages",
    y=0.97,
)
figure.subplots_adjust(
    left=0.06,
    right=0.91,
    bottom=0.04,
    top=0.94,
    hspace=0.14,
)
figure.savefig(
    figure_root / "figure_5_three_age_3d.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.25,
)
plt.show()
plt.close(figure)

# Orthogonal projections make the geometry auditable without relying
# on a single 3D viewing angle.
figure, axes = plt.subplots(
    len(ABSOLUTE_CLOCKS),
    3,
    figsize=(15.5, 9),
    squeeze=False,
    layout="constrained",
)
for row, (clock_column, clock_label) in enumerate(
    ABSOLUTE_CLOCKS.items()
):
    subset = three_age_plot_data[clock_column]
    panels = (
        (
            "chronological_age",
            "retinal_age_oof",
            "Chronological age",
            "Retinal age",
        ),
        (
            "chronological_age",
            clock_column,
            "Chronological age",
            clock_label,
        ),
        (
            "retinal_age_oof",
            clock_column,
            "Retinal age",
            clock_label,
        ),
    )
    shared_lower = float(
        subset[
            ["chronological_age", "retinal_age_oof", clock_column]
        ].min().min()
    )
    shared_upper = float(
        subset[
            ["chronological_age", "retinal_age_oof", clock_column]
        ].max().max()
    )
    for column, (
        x_column,
        y_column,
        x_label,
        y_label,
    ) in enumerate(panels):
        axis = axes[row, column]
        axis.hexbin(
            subset[x_column],
            subset[y_column],
            gridsize=38,
            mincnt=1,
            cmap="viridis",
        )
        axis.plot(
            [shared_lower, shared_upper],
            [shared_lower, shared_upper],
            "--",
            color="black",
            linewidth=1,
        )
        axis.set_xlim(shared_lower, shared_upper)
        axis.set_ylim(shared_lower, shared_upper)
        axis.set_aspect("equal", adjustable="box")
        axis.set(
            title=f"{x_label} versus {y_label}",
            xlabel=f"{x_label} (years)",
            ylabel=f"{y_label} (years)",
        )
        if column == 0:
            axis.text(
                -0.23,
                1.10,
                clock_label,
                transform=axis.transAxes,
                fontsize=12,
                fontweight="semibold",
                ha="left",
            )
figure.suptitle(
    "Two-dimensional projections of the three-age relationship",
    y=1.02,
)
figure.savefig(
    figure_root / "figure_5b_three_age_pairwise_projections.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.25,
)
plt.show()
plt.close(figure)

figure, axes = plt.subplots(
    len(ABSOLUTE_CLOCKS),
    1,
    figsize=(10.5, 8.5),
    squeeze=False,
    layout="constrained",
)
for axis, (clock_column, clock_label) in zip(
    axes.ravel(),
    ABSOLUTE_CLOCKS.items(),
):
    plot = distance_summary[
        distance_summary["clock"].eq(clock_column)
    ].copy()
    point = plot["mean_absolute_distance"].to_numpy(float)
    low = plot["ci_low"].to_numpy(float)
    high = plot["ci_high"].to_numpy(float)
    xerr = np.vstack(
        [
            np.maximum(
                point - np.minimum(low, point),
                0.0,
            ),
            np.maximum(
                np.maximum(high, point) - point,
                0.0,
            ),
        ]
    )
    position = np.arange(len(plot))
    axis.errorbar(
        point,
        position,
        xerr=xerr,
        fmt="o",
        markersize=7,
        color="#4C72B0",
        ecolor="#4C72B0",
        elinewidth=2,
        capsize=4,
    )
    axis.set_yticks(
        position,
        plot["pair"],
    )
    axis.set(
        title=clock_label,
        xlabel="Mean absolute age difference (years; 95% CI)",
        ylabel="",
    )
    axis.set_xlim(left=0)
    axis.invert_yaxis()
figure.suptitle(
    "Pairwise distances among chronological, retinal, and epigenetic ages",
    y=1.02,
)
figure.savefig(
    figure_root / "figure_6_three_age_absolute_distances.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.25,
)
plt.show()
plt.close(figure)

## 10. Central-age and tail sensitivity

The same paired distance hypotheses are repeated in the prespecified
central 80%, lower 10%, and upper 10% chronological-age strata. These
results reveal whether apparent agreement is concentrated in the age range
where the chronological RETFound model is best calibrated.

In [ ]:
master["age_distribution_stratum"] = np.select(
    [
        master["chronological_age"] < age_q10,
        master["chronological_age"] > age_q90,
    ],
    ["Lower 10% age tail", "Upper 10% age tail"],
    default="Central 80%",
)
sensitivity_rows = []
for stratum_index, (stratum, stratum_frame) in enumerate(
    master.groupby("age_distribution_stratum")
):
    for clock_index, (clock_column, clock_label) in enumerate(
        ABSOLUTE_CLOCKS.items()
    ):
        subset = stratum_frame.dropna(
            subset=[
                "chronological_age",
                "retinal_age_oof",
                clock_column,
            ]
        )
        if len(subset) < minimum_reporting_group_n:
            continue
        retinal_epigenetic = np.abs(
            subset["retinal_age_oof"] - subset[clock_column]
        ).to_numpy(float)
        comparators = {
            "Retinal–chronological": np.abs(
                subset["retinal_age_oof"]
                - subset["chronological_age"]
            ).to_numpy(float),
            "Epigenetic–chronological": np.abs(
                subset[clock_column]
                - subset["chronological_age"]
            ).to_numpy(float),
        }
        for comparison_index, (
            comparator_label,
            comparator,
        ) in enumerate(comparators.items()):
            difference = retinal_epigenetic - comparator
            low, high = bootstrap_mean_ci(
                difference,
                bootstrap_repetitions,
                random_state
                + 9000
                + 1000 * stratum_index
                + 100 * clock_index
                + comparison_index,
            )
            sensitivity_rows.append(
                {
                    "age_distribution_stratum": stratum,
                    "clock": clock_column,
                    "clock_label": clock_label,
                    "comparator": comparator_label,
                    "n": int(len(subset)),
                    "mean_distance_difference": float(
                        np.mean(difference)
                    ),
                    "ci_low": low,
                    "ci_high": high,
                    "proportion_retinal_epigenetic_smaller": float(
                        np.mean(difference < 0)
                    ),
                    "signflip_one_sided_p": signflip_less_test(
                        difference,
                        permutation_repetitions,
                        random_state
                        + 12000
                        + 1000 * stratum_index
                        + 100 * clock_index
                        + comparison_index,
                    ),
                }
            )
age_stratified_distance_tests = pd.DataFrame(sensitivity_rows)
if not age_stratified_distance_tests.empty:
    age_stratified_distance_tests["signflip_fdr_q"] = (
        benjamini_hochberg(
            age_stratified_distance_tests[
                "signflip_one_sided_p"
            ].to_numpy(float)
        )
    )
    stratum_bias = (
        tail_performance.set_index("age_distribution_stratum")[
            "mean_error"
        ]
    )
    age_stratified_distance_tests[
        "retinal_model_mean_error"
    ] = age_stratified_distance_tests[
        "age_distribution_stratum"
    ].map(stratum_bias)
    age_stratified_distance_tests[
        "supports_closer_statistically"
    ] = (
        age_stratified_distance_tests["ci_high"].lt(0)
        & age_stratified_distance_tests[
            "signflip_fdr_q"
        ].lt(0.05)
    )
    age_stratified_distance_tests[
        "tail_calibration_warning"
    ] = (
        age_stratified_distance_tests[
            "age_distribution_stratum"
        ].ne("Central 80%")
        & age_stratified_distance_tests[
            "retinal_model_mean_error"
        ].abs().ge(2.0)
    )
    age_stratified_distance_tests[
        "publication_support_flag"
    ] = (
        age_stratified_distance_tests[
            "supports_closer_statistically"
        ]
        & ~age_stratified_distance_tests[
            "tail_calibration_warning"
        ]
    )
    age_stratified_distance_tests["interpretation"] = np.select(
        [
            age_stratified_distance_tests[
                "supports_closer_statistically"
            ]
            & age_stratified_distance_tests[
                "tail_calibration_warning"
            ],
            age_stratified_distance_tests[
                "supports_closer_statistically"
            ],
        ],
        [
            (
                "Statistically positive but calibration-sensitive: "
                "do not interpret as biological convergence"
            ),
            "Statistically positive within the calibrated age range",
        ],
        default="No evidence that retinal–epigenetic distance is smaller",
    )
write_frame(
    age_stratified_distance_tests,
    statistics_root / "age_stratified_three_age_distance_tests.csv",
)
display(age_stratified_distance_tests.round(5))

## 11. Patient-level three-clock acceleration and discordance

Raw age gaps inherit regression-to-the-mean from each clock. This section
therefore fits age splines in held-out participants and defines
acceleration as the cross-fitted residual from the expected clock value
at that chronological age. Residuals are standardized before combining
clocks. Continuous phenotypes are primary; percentile groups are
descriptive only.

In [ ]:
tri_age = master.dropna(
    subset=[
        "participant_id",
        "chronological_age",
        "retinal_age_oof",
        "epigenetic_dnam_age",
        "epigenetic_hannum_age",
    ]
).copy()
if tri_age["participant_id"].duplicated().any():
    raise ValueError("Three-clock cohort must be participant-unique")

clock_definitions = {
    "retinal": "retinal_age_oof",
    "horvath": "epigenetic_dnam_age",
    "hannum": "epigenetic_hannum_age",
}
for clock_name, estimate_column in clock_definitions.items():
    residual, expected = cross_fitted_spline_residual(
        tri_age,
        estimate_column,
        "chronological_age",
        "participant_id",
    )
    tri_age[f"{clock_name}_age_expected_cf"] = expected
    tri_age[f"{clock_name}_acceleration_cf"] = residual
    tri_age[f"z_{clock_name}_acceleration"] = safe_zscore(
        residual
    )

acceleration_columns = [
    "z_retinal_acceleration",
    "z_horvath_acceleration",
    "z_hannum_acceleration",
]
tri_age["epigenetic_mean_acceleration"] = tri_age[
    ["z_horvath_acceleration", "z_hannum_acceleration"]
].mean(axis=1)
tri_age["z_epigenetic_mean_acceleration"] = safe_zscore(
    tri_age["epigenetic_mean_acceleration"]
)
tri_age["shared_acceleration_mean"] = tri_age[
    acceleration_columns
].mean(axis=1)
tri_age["z_shared_acceleration_mean"] = safe_zscore(
    tri_age["shared_acceleration_mean"]
)
tri_age["retina_specific_discordance"] = safe_zscore(
    tri_age["z_retinal_acceleration"]
    - tri_age["epigenetic_mean_acceleration"]
)
tri_age["epigenetic_specific_discordance"] = (
    -tri_age["retina_specific_discordance"]
)
tri_age["three_clock_dispersion"] = tri_age[
    acceleration_columns
].std(axis=1, ddof=1)
tri_age["z_three_clock_dispersion"] = safe_zscore(
    tri_age["three_clock_dispersion"]
)
tri_age["horvath_hannum_discordance"] = safe_zscore(
    tri_age["z_horvath_acceleration"]
    - tri_age["z_hannum_acceleration"]
)

acceleration_matrix = tri_age[
    acceleration_columns
].to_numpy(float)
pca = PCA(n_components=3, random_state=random_state)
raw_scores = pca.fit_transform(acceleration_matrix)
oriented_loadings = pca.components_.copy()
score_signs = np.ones(3)

shared_index = int(
    np.argmax(np.abs(oriented_loadings.sum(axis=1)))
)
if oriented_loadings[shared_index].mean() < 0:
    score_signs[shared_index] = -1
remaining = [index for index in range(3) if index != shared_index]
retina_index = max(
    remaining,
    key=lambda index: abs(
        oriented_loadings[index, 0]
        - oriented_loadings[index, 1:].mean()
    ),
)
if (
    oriented_loadings[retina_index, 0]
    - oriented_loadings[retina_index, 1:].mean()
) < 0:
    score_signs[retina_index] = -1
epigenetic_index = next(
    index
    for index in range(3)
    if index not in {shared_index, retina_index}
)
if (
    oriented_loadings[epigenetic_index, 1]
    - oriented_loadings[epigenetic_index, 2]
) < 0:
    score_signs[epigenetic_index] = -1

oriented_loadings = oriented_loadings * score_signs[:, None]
oriented_scores = raw_scores * score_signs[None, :]
component_roles = {
    "shared_aging_pc": shared_index,
    "retina_vs_epigenetic_pc": retina_index,
    "horvath_vs_hannum_pc": epigenetic_index,
}
for role, component_index in component_roles.items():
    tri_age[role] = safe_zscore(
        pd.Series(
            oriented_scores[:, component_index],
            index=tri_age.index,
        )
    )

pca_rows = []
for role, component_index in component_roles.items():
    pca_rows.append(
        {
            "component_role": role,
            "original_component_number": component_index + 1,
            "explained_variance_ratio": float(
                pca.explained_variance_ratio_[component_index]
            ),
            **{
                f"loading_{name}": float(
                    oriented_loadings[component_index, position]
                )
                for position, name in enumerate(
                    ("retinal", "horvath", "hannum")
                )
            },
        }
    )
pca_loadings = pd.DataFrame(pca_rows)
write_frame(
    pca_loadings,
    statistics_root / "three_clock_pca_loadings.csv",
)

low_thresholds = tri_age[
    acceleration_columns
].quantile(1 - acceleration_extreme_quantile)
high_thresholds = tri_age[
    acceleration_columns
].quantile(acceleration_extreme_quantile)
retinal_high = (
    tri_age["z_retinal_acceleration"]
    >= high_thresholds["z_retinal_acceleration"]
)
retinal_low = (
    tri_age["z_retinal_acceleration"]
    <= low_thresholds["z_retinal_acceleration"]
)
epigenetic_high = (
    tri_age["z_horvath_acceleration"]
    >= high_thresholds["z_horvath_acceleration"]
) & (
    tri_age["z_hannum_acceleration"]
    >= high_thresholds["z_hannum_acceleration"]
)
epigenetic_low = (
    tri_age["z_horvath_acceleration"]
    <= low_thresholds["z_horvath_acceleration"]
) & (
    tri_age["z_hannum_acceleration"]
    <= low_thresholds["z_hannum_acceleration"]
)
tri_age["three_clock_profile"] = np.select(
    [
        retinal_high & epigenetic_high,
        retinal_low & epigenetic_low,
        retinal_high & ~epigenetic_high,
        ~retinal_high & epigenetic_high,
    ],
    [
        "Concordantly accelerated",
        "Concordantly decelerated",
        "Retina-only accelerated",
        "Epigenetic-only accelerated",
    ],
    default="Mixed or intermediate",
)

COMORBIDITY_LABELS = {
    "diabetes": "Diabetes",
    "hypertension": "Hypertension",
    "heart_disease": "Heart disease",
    "stroke": "Stroke",
    "kidney_disease": "Kidney disease",
    "chronic_kidney_disease": "Chronic kidney disease",
    "arthritis_any": "Arthritis",
    "osteoporosis": "Osteoporosis",
    "asthma_or_copd": "Asthma or COPD",
    "cancer": "Cancer",
    "low_back_pain": "Low back pain",
}
available_binary_comorbidities = {}
binary_columns = []
for source_column, label in COMORBIDITY_LABELS.items():
    if source_column not in tri_age.columns:
        continue
    binary_column = f"{source_column}_binary"
    tri_age[binary_column] = coerce_binary(
        tri_age[source_column]
    )
    if tri_age[binary_column].notna().sum() == 0:
        continue
    available_binary_comorbidities[binary_column] = label
    binary_columns.append(binary_column)
tri_age["comorbidity_observed_count"] = tri_age[
    binary_columns
].notna().sum(axis=1)
tri_age["comorbidity_burden"] = tri_age[
    binary_columns
].sum(axis=1, min_count=1)
tri_age["comorbidity_fraction"] = (
    tri_age["comorbidity_burden"]
    / tri_age["comorbidity_observed_count"].replace(0, np.nan)
)

dispersion_cutoff = tri_age[
    "three_clock_dispersion"
].quantile(0.95)
retina_specific_low = tri_age[
    "retina_specific_discordance"
].quantile(0.05)
retina_specific_high = tri_age[
    "retina_specific_discordance"
].quantile(0.95)
tri_age["extreme_discordance_group"] = np.select(
    [
        tri_age["three_clock_dispersion"] >= dispersion_cutoff,
        tri_age["retina_specific_discordance"]
        <= retina_specific_low,
        tri_age["retina_specific_discordance"]
        >= retina_specific_high,
    ],
    [
        "Highest 5% overall disagreement",
        "Retina younger than epigenetic clocks",
        "Retina older than epigenetic clocks",
    ],
    default="Not extreme",
)
write_frame(
    tri_age,
    private_root / "patient_three_clock_phenotypes_private.parquet",
)
write_frame(
    tri_age[
        tri_age["extreme_discordance_group"].ne("Not extreme")
    ],
    private_root
    / "extreme_discordance_explainability_cohort_private.parquet",
)
profile_summary = (
    tri_age.groupby("three_clock_profile", as_index=False)
    .agg(
        participants=("participant_id", "nunique"),
        mean_shared_acceleration=("shared_acceleration_mean", "mean"),
        mean_retina_specific=(
            "retina_specific_discordance",
            "mean",
        ),
        mean_clock_dispersion=("three_clock_dispersion", "mean"),
        mean_comorbidity_burden=("comorbidity_burden", "mean"),
    )
)
profile_comorbidity_rows = []
for outcome_column, outcome_label in (
    available_binary_comorbidities.items()
):
    for profile, subset in tri_age.groupby(
        "three_clock_profile"
    ):
        observed = subset[outcome_column].dropna()
        if observed.empty:
            continue
        profile_comorbidity_rows.append(
            {
                "three_clock_profile": profile,
                "outcome": outcome_column,
                "outcome_label": outcome_label,
                "n_observed": int(len(observed)),
                "events": int(observed.eq(1).sum()),
                "prevalence": float(observed.mean()),
            }
        )
profile_comorbidity_summary = pd.DataFrame(
    profile_comorbidity_rows
)
write_frame(
    profile_summary,
    statistics_root / "three_clock_profile_summary.csv",
)
write_frame(
    profile_comorbidity_summary,
    statistics_root
    / "three_clock_profile_comorbidity_prevalence.csv",
)
display(pca_loadings.round(4))
display(profile_summary.round(4))

figure = plt.figure(figsize=(14, 11), layout="constrained")
grid = figure.add_gridspec(2, 2, height_ratios=[1, 1.15])
scatter_axis = figure.add_subplot(grid[0, 0])
violin_axis = figure.add_subplot(grid[0, 1])
heatmap_axis = figure.add_subplot(grid[1, :])

scatter = scatter_axis.scatter(
    tri_age["z_epigenetic_mean_acceleration"],
    tri_age["z_retinal_acceleration"],
    c=tri_age["three_clock_dispersion"],
    cmap="magma",
    s=18,
    alpha=0.65,
)
scatter_axis.axhline(0, color="black", linewidth=1)
scatter_axis.axvline(0, color="black", linewidth=1)
scatter_axis.set(
    title="Within-participant retinal–epigenetic concordance",
    xlabel="Mean epigenetic acceleration (SD)",
    ylabel="Retinal acceleration (SD)",
)
figure.colorbar(
    scatter,
    ax=scatter_axis,
    label="Three-clock dispersion",
    shrink=0.85,
)

acceleration_long = tri_age[
    acceleration_columns
].rename(
    columns={
        "z_retinal_acceleration": "Retinal",
        "z_horvath_acceleration": "Horvath",
        "z_hannum_acceleration": "Hannum",
    }
).melt(var_name="Clock", value_name="Acceleration")
sns.violinplot(
    data=acceleration_long,
    x="Clock",
    y="Acceleration",
    inner="quartile",
    cut=0,
    ax=violin_axis,
)
violin_axis.axhline(0, color="black", linewidth=1)
violin_axis.set(
    title="Cross-fitted acceleration distributions",
    xlabel="",
    ylabel="Acceleration (SD)",
)

heatmap_data = (
    tri_age.nlargest(60, "three_clock_dispersion")[
        acceleration_columns
    ]
    .rename(
        columns={
            "z_retinal_acceleration": "Retinal",
            "z_horvath_acceleration": "Horvath",
            "z_hannum_acceleration": "Hannum",
        }
    )
    .apply(pd.to_numeric, errors="coerce")
    .astype("float64")
    .replace([np.inf, -np.inf], np.nan)
    .dropna(axis=0, how="any")
    .reset_index(drop=True)
)
if not heatmap_data.empty:
    sns.heatmap(
        heatmap_data.to_numpy(dtype=float),
        cmap="vlag",
        center=0,
        yticklabels=False,
        xticklabels=heatmap_data.columns.tolist(),
        ax=heatmap_axis,
        cbar_kws={"label": "Acceleration (SD)"},
    )
    heatmap_axis.set(
        title=(
            "Sixty most discordant participants "
            "(identifiers suppressed)"
        ),
        xlabel="",
        ylabel="Participants ordered by discordance",
    )
else:
    heatmap_axis.text(
        0.5,
        0.5,
        "No complete numeric acceleration records available",
        ha="center",
        va="center",
        transform=heatmap_axis.transAxes,
    )
    heatmap_axis.set_axis_off()
figure.suptitle(
    "Patient-level three-clock acceleration phenotypes",
    y=1.02,
)
figure.savefig(
    figure_root / "figure_7_patient_three_clock_phenotypes.png",
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.25,
)
plt.show()
plt.close(figure)

## 12. Comorbidity associations and unique clock information

Binary diseases are treated as outcomes. The primary mutually adjusted
model includes retinal and mean epigenetic acceleration together, while
an orthogonal PCA model separates shared aging, retina-versus-epigenetic
discordance, and Horvath-versus-Hannum discordance. Associations are
cross-sectional and cannot establish causation.

In [ ]:
association_covariate_terms = [
    "chronological_age",
    "chronological_age_sq",
]
tri_age["chronological_age_sq"] = tri_age[
    "chronological_age"
] ** 2
tri_age["log_n_embedded_images"] = np.log1p(
    pd.to_numeric(
        tri_age["n_embedded_images"],
        errors="coerce",
    ).fillna(0)
)
association_covariate_terms.append("log_n_embedded_images")
association_categorical_columns = []
for categorical in (
    "sex_labeled",
    "racial_background",
    "smoking_status",
):
    if categorical not in tri_age.columns:
        continue
    counts = tri_age[categorical].value_counts(dropna=True)
    if len(counts) < 2:
        continue
    frequent = counts[counts >= minimum_reporting_group_n].index
    model_column = f"{categorical}_association"
    category_source = tri_age[categorical].astype("string")
    tri_age[model_column] = (
        category_source
        .where(
            category_source.isna()
            | category_source.isin(frequent.astype(str)),
            "Other/low-frequency",
        )
        .fillna("Missing")
        .astype(str)
    )
    tri_age[model_column] = pd.Categorical(
        tri_age[model_column]
    )
    association_categorical_columns.append(model_column)
    association_covariate_terms.append(f"C({model_column})")

for continuous in ("bmi", "body_mass_index"):
    if continuous in tri_age.columns:
        tri_age[continuous] = pd.to_numeric(
            tri_age[continuous], errors="coerce"
        )
        if tri_age[continuous].notna().sum() >= 100:
            association_covariate_terms.append(continuous)
            break

clock_predictor_sets = {
    "mutually_adjusted_clocks": [
        "z_retinal_acceleration",
        "z_epigenetic_mean_acceleration",
    ],
    "orthogonal_three_clock_components": [
        "shared_aging_pc",
        "retina_vs_epigenetic_pc",
        "horvath_vs_hannum_pc",
    ],
    "direct_clock_discordance": [
        "z_shared_acceleration_mean",
        "retina_specific_discordance",
        "horvath_hannum_discordance",
        "z_three_clock_dispersion",
    ],
}
association_rows = []
association_contrasts = []
association_failures = []
for outcome_index, (
    outcome_column,
    outcome_label,
) in enumerate(available_binary_comorbidities.items()):
    outcome = pd.to_numeric(
        tri_age[outcome_column], errors="coerce"
    )
    event_count = int(outcome.eq(1).sum())
    nonevent_count = int(outcome.eq(0).sum())
    if min(event_count, nonevent_count) < minimum_binary_outcome_events:
        association_failures.append(
            {
                "outcome": outcome_column,
                "outcome_label": outcome_label,
                "status": "skipped_insufficient_events",
                "events": event_count,
                "non_events": nonevent_count,
            }
        )
        continue
    for framework, predictors in clock_predictor_sets.items():
        formula = (
            f"{outcome_column} ~ "
            + " + ".join(
                [*association_covariate_terms, *predictors]
            )
        )
        try:
            fit = smf.glm(
                formula,
                data=tri_age,
                family=sm.families.Binomial(),
                missing="drop",
            ).fit(cov_type="HC3")
            confidence = fit.conf_int()
            for predictor in predictors:
                association_rows.append(
                    {
                        "outcome": outcome_column,
                        "outcome_label": outcome_label,
                        "framework": framework,
                        "predictor": predictor,
                        "n": int(fit.nobs),
                        "events": event_count,
                        "log_odds_coefficient": float(
                            fit.params[predictor]
                        ),
                        "odds_ratio": float(
                            np.exp(fit.params[predictor])
                        ),
                        "ci_low": float(
                            np.exp(confidence.loc[predictor, 0])
                        ),
                        "ci_high": float(
                            np.exp(confidence.loc[predictor, 1])
                        ),
                        "p_value": float(
                            fit.pvalues[predictor]
                        ),
                    }
                )
            if framework == "mutually_adjusted_clocks":
                contrast = fit.t_test(
                    "z_retinal_acceleration "
                    "- z_epigenetic_mean_acceleration = 0"
                )
                association_contrasts.append(
                    {
                        "outcome": outcome_column,
                        "outcome_label": outcome_label,
                        "n": int(fit.nobs),
                        "events": event_count,
                        "retinal_minus_epigenetic_log_odds": float(
                            np.asarray(contrast.effect).reshape(-1)[0]
                        ),
                        "p_value": float(contrast.pvalue),
                    }
                )
        except Exception as error:
            association_failures.append(
                {
                    "outcome": outcome_column,
                    "outcome_label": outcome_label,
                    "framework": framework,
                    "status": f"failed:{type(error).__name__}",
                    "error_message": str(error)[:500],
                    "events": event_count,
                    "non_events": nonevent_count,
                }
            )

comorbidity_associations = pd.DataFrame(association_rows)
comorbidity_clock_contrasts = pd.DataFrame(
    association_contrasts
)
comorbidity_association_failures = pd.DataFrame(
    association_failures
)
if not comorbidity_associations.empty:
    comorbidity_associations["fdr_q_value"] = (
        benjamini_hochberg(
            comorbidity_associations["p_value"].to_numpy(float)
        )
    )
    comorbidity_associations["significant_fdr_0_05"] = (
        comorbidity_associations["fdr_q_value"] < 0.05
    )
if not comorbidity_clock_contrasts.empty:
    comorbidity_clock_contrasts["fdr_q_value"] = (
        benjamini_hochberg(
            comorbidity_clock_contrasts[
                "p_value"
            ].to_numpy(float)
        )
    )

multimorbidity_rows = []
if tri_age["comorbidity_burden"].notna().sum() >= 100:
    for framework, predictors in clock_predictor_sets.items():
        formula = (
            "comorbidity_burden ~ "
            + " + ".join(
                [*association_covariate_terms, *predictors]
            )
        )
        try:
            fit = smf.glm(
                formula,
                data=tri_age,
                family=sm.families.NegativeBinomial(alpha=1.0),
                missing="drop",
            ).fit(cov_type="HC3")
            confidence = fit.conf_int()
            for predictor in predictors:
                multimorbidity_rows.append(
                    {
                        "framework": framework,
                        "predictor": predictor,
                        "n": int(fit.nobs),
                        "rate_ratio": float(
                            np.exp(fit.params[predictor])
                        ),
                        "ci_low": float(
                            np.exp(confidence.loc[predictor, 0])
                        ),
                        "ci_high": float(
                            np.exp(confidence.loc[predictor, 1])
                        ),
                        "p_value": float(
                            fit.pvalues[predictor]
                        ),
                    }
                )
        except Exception as error:
            association_failures.append(
                {
                    "outcome": "comorbidity_burden",
                    "outcome_label": "Comorbidity burden",
                    "framework": framework,
                    "status": f"failed:{type(error).__name__}",
                    "error_message": str(error)[:500],
                }
            )
multimorbidity_associations = pd.DataFrame(multimorbidity_rows)
if not multimorbidity_associations.empty:
    multimorbidity_associations["fdr_q_value"] = (
        benjamini_hochberg(
            multimorbidity_associations[
                "p_value"
            ].to_numpy(float)
        )
    )

write_frame(
    comorbidity_associations,
    statistics_root / "patient_clock_comorbidity_associations.csv",
)
write_frame(
    comorbidity_clock_contrasts,
    statistics_root / "retinal_vs_epigenetic_effect_contrasts.csv",
)
write_frame(
    pd.DataFrame(association_failures),
    statistics_root / "comorbidity_association_failures.csv",
)
write_frame(
    multimorbidity_associations,
    statistics_root / "multimorbidity_clock_associations.csv",
)
display(comorbidity_associations.round(5))
display(multimorbidity_associations.round(5))

if not comorbidity_associations.empty:
    forest_data = comorbidity_associations[
        comorbidity_associations["framework"].eq(
            "mutually_adjusted_clocks"
        )
    ].copy()
    figure, axes = plt.subplots(
        2,
        1,
        figsize=(11, max(8, 0.55 * len(forest_data))),
        layout="constrained",
    )
    predictor_labels = {
        "z_retinal_acceleration": "Retinal acceleration",
        "z_epigenetic_mean_acceleration": (
            "Mean epigenetic acceleration"
        ),
    }
    for axis, predictor in zip(
        axes,
        predictor_labels,
    ):
        plot = forest_data[
            forest_data["predictor"].eq(predictor)
        ].sort_values("odds_ratio")
        point = plot["odds_ratio"].to_numpy(float)
        low = plot["ci_low"].to_numpy(float)
        high = plot["ci_high"].to_numpy(float)
        error = np.vstack(
            [
                np.maximum(point - low, 0),
                np.maximum(high - point, 0),
            ]
        )
        positions = np.arange(len(plot))
        colors = np.where(
            plot["significant_fdr_0_05"],
            "#C44E52",
            "#4C72B0",
        )
        for position, estimate, interval, color in zip(
            positions,
            point,
            error.T,
            colors,
        ):
            axis.errorbar(
                estimate,
                position,
                xerr=interval.reshape(2, 1),
                fmt="o",
                color=color,
                capsize=3,
            )
        axis.axvline(1, color="black", linestyle="--")
        axis.set_xscale("log")
        axis.set_yticks(
            positions,
            plot["outcome_label"],
        )
        axis.set(
            title=predictor_labels[predictor],
            xlabel="Adjusted odds ratio per 1-SD increase (95% CI)",
            ylabel="",
        )
    figure.suptitle(
        "Independent associations of aging clocks with comorbidities",
        y=1.02,
    )
    figure.savefig(
        figure_root
        / "figure_8_clock_comorbidity_association_forest.png",
        dpi=220,
        bbox_inches="tight",
        pad_inches=0.25,
    )
    plt.show()
    plt.close(figure)

## 13. Incremental disease-prediction value

Five-fold stratified out-of-fold predictions compare chronological and
demographic covariates alone with retinal, epigenetic, combined, and
orthogonal three-clock models. This evaluates incremental information
without reporting optimistic in-sample AUC.

In [ ]:
prediction_continuous_base = [
    "chronological_age",
    "chronological_age_sq",
    "log_n_embedded_images",
]
prediction_categorical_base = association_categorical_columns
prediction_model_sets = {
    "base": [],
    "base_plus_retinal": ["z_retinal_acceleration"],
    "base_plus_epigenetic": [
        "z_epigenetic_mean_acceleration"
    ],
    "base_plus_both": [
        "z_retinal_acceleration",
        "z_epigenetic_mean_acceleration",
    ],
    "base_plus_orthogonal_components": [
        "shared_aging_pc",
        "retina_vs_epigenetic_pc",
        "horvath_vs_hannum_pc",
    ],
    "base_plus_direct_discordance": [
        "z_shared_acceleration_mean",
        "retina_specific_discordance",
        "horvath_hannum_discordance",
        "z_three_clock_dispersion",
    ],
}

def cross_validated_binary_prediction(
    frame,
    outcome_column,
    additional_continuous,
    seed,
):
    continuous = [
        *prediction_continuous_base,
        *additional_continuous,
    ]
    columns = [
        outcome_column,
        *continuous,
        *prediction_categorical_base,
    ]
    work = frame[columns].copy()
    work[outcome_column] = pd.to_numeric(
        work[outcome_column], errors="coerce"
    )
    work = work.dropna(subset=[outcome_column])
    outcome = work[outcome_column].astype(int).to_numpy()
    event_count = int(outcome.sum())
    nonevent_count = int(len(outcome) - event_count)
    splits = min(5, event_count, nonevent_count)
    if splits < 2:
        raise ValueError("Insufficient events for stratified CV")
    transformers = [
        (
            "continuous",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            continuous,
        )
    ]
    if prediction_categorical_base:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(
                                strategy="most_frequent"
                            ),
                        ),
                        (
                            "one_hot",
                            OneHotEncoder(
                                handle_unknown="ignore"
                            ),
                        ),
                    ]
                ),
                prediction_categorical_base,
            )
        )
    estimator = Pipeline(
        [
            (
                "preprocess",
                ColumnTransformer(transformers),
            ),
            (
                "logistic",
                LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    solver="liblinear",
                    max_iter=2000,
                    random_state=seed,
                ),
            ),
        ]
    )
    splitter = StratifiedKFold(
        n_splits=splits,
        shuffle=True,
        random_state=seed,
    )
    prediction = cross_val_predict(
        estimator,
        work.drop(columns=[outcome_column]),
        outcome,
        cv=splitter,
        method="predict_proba",
    )[:, 1]
    return work.index, outcome, prediction

predictive_metric_rows = []
predictive_prediction_frames = []
for outcome_index, (
    outcome_column,
    outcome_label,
) in enumerate(available_binary_comorbidities.items()):
    outcome = pd.to_numeric(
        tri_age[outcome_column], errors="coerce"
    )
    if min(
        int(outcome.eq(1).sum()),
        int(outcome.eq(0).sum()),
    ) < minimum_binary_outcome_events:
        continue
    model_predictions = {}
    common_index = None
    common_outcome = None
    for model_index, (
        model_name,
        added_predictors,
    ) in enumerate(prediction_model_sets.items()):
        index, y, prediction = cross_validated_binary_prediction(
            tri_age,
            outcome_column,
            added_predictors,
            random_state + 100 * outcome_index,
        )
        if common_index is None:
            common_index = index
            common_outcome = y
        elif not index.equals(common_index):
            raise RuntimeError(
                "Nested prediction models used different participants"
            )
        model_predictions[model_name] = prediction
        predictive_metric_rows.append(
            {
                "outcome": outcome_column,
                "outcome_label": outcome_label,
                "model": model_name,
                "n": int(len(y)),
                "events": int(y.sum()),
                "oof_auc": float(roc_auc_score(y, prediction)),
                "oof_brier": float(
                    brier_score_loss(y, prediction)
                ),
            }
        )
    base_prediction = model_predictions["base"]
    for model_index, (
        model_name,
        prediction,
    ) in enumerate(model_predictions.items()):
        if model_name == "base":
            continue
        auc_low, auc_high = bootstrap_metric_difference(
            common_outcome,
            prediction,
            base_prediction,
            roc_auc_score,
            prediction_bootstrap_repetitions,
            random_state
            + 10000
            + 100 * outcome_index
            + model_index,
        )
        brier_low, brier_high = bootstrap_metric_difference(
            common_outcome,
            prediction,
            base_prediction,
            lambda observed, predicted: -brier_score_loss(
                observed, predicted
            ),
            prediction_bootstrap_repetitions,
            random_state
            + 20000
            + 100 * outcome_index
            + model_index,
        )
        predictive_metric_rows.append(
            {
                "outcome": outcome_column,
                "outcome_label": outcome_label,
                "model": f"{model_name}_increment_vs_base",
                "n": int(len(common_outcome)),
                "events": int(common_outcome.sum()),
                "oof_auc": float(
                    roc_auc_score(
                        common_outcome,
                        prediction,
                    )
                ),
                "delta_auc_vs_base": float(
                    roc_auc_score(
                        common_outcome,
                        prediction,
                    )
                    - roc_auc_score(
                        common_outcome,
                        base_prediction,
                    )
                ),
                "delta_auc_ci_low": auc_low,
                "delta_auc_ci_high": auc_high,
                "brier_improvement_vs_base": float(
                    brier_score_loss(
                        common_outcome,
                        base_prediction,
                    )
                    - brier_score_loss(
                        common_outcome,
                        prediction,
                    )
                ),
                "brier_improvement_ci_low": brier_low,
                "brier_improvement_ci_high": brier_high,
            }
        )
    predictive_prediction_frames.append(
        pd.DataFrame(
            {
                "participant_id": tri_age.loc[
                    common_index,
                    "participant_id",
                ].astype(str).to_numpy(),
                "outcome": outcome_column,
                "observed": common_outcome,
                **model_predictions,
            }
        )
    )

predictive_metrics = pd.DataFrame(predictive_metric_rows)
predictive_predictions = (
    pd.concat(
        predictive_prediction_frames,
        ignore_index=True,
    )
    if predictive_prediction_frames
    else pd.DataFrame()
)
write_frame(
    predictive_metrics,
    statistics_root
    / "comorbidity_incremental_prediction_metrics.csv",
)
write_frame(
    predictive_predictions,
    private_root
    / "comorbidity_oof_predictions_private.parquet",
)
display(predictive_metrics.round(5))

increment_plot = (
    predictive_metrics[
        predictive_metrics["model"].str.endswith(
            "_increment_vs_base",
            na=False,
        )
    ].copy()
    if not predictive_metrics.empty
    else pd.DataFrame()
)
if not increment_plot.empty:
    increment_plot["model"] = (
        increment_plot["model"]
        .str.replace("_increment_vs_base", "", regex=False)
        .str.replace("base_plus_", "", regex=False)
        .str.replace("_", " ")
    )
    heatmap = increment_plot.pivot(
        index="outcome_label",
        columns="model",
        values="delta_auc_vs_base",
    )
    figure, axis = plt.subplots(
        figsize=(
            max(9, 1.8 * heatmap.shape[1]),
            max(5, 0.6 * heatmap.shape[0]),
        ),
        layout="constrained",
    )
    sns.heatmap(
        heatmap,
        annot=True,
        fmt=".3f",
        center=0,
        cmap="vlag",
        ax=axis,
        cbar_kws={"label": "OOF ΔAUC versus base"},
    )
    axis.set(
        title="Incremental comorbidity discrimination from aging clocks",
        xlabel="Added clock phenotype",
        ylabel="",
    )
    figure.savefig(
        figure_root
        / "figure_9_comorbidity_incremental_auc.png",
        dpi=220,
        bbox_inches="tight",
        pad_inches=0.25,
    )
    plt.show()
    plt.close(figure)

## 14. Questionnaire-wide phenome scan

Every field in the raw baseline CoPv7_Qx_CANUE_PA_BS questionnaire CSV
is loaded for the three-clock cohort in resumable row batches.
Identifiers, dates, age leakage,
epigenetic variables, demographics already modeled separately, and the
comorbidities analyzed above are retained in the private extract but
excluded from this scan. Numeric questions receive standardized linear
effects; categorical questions receive omnibus nested-model tests.
False-discovery control is applied globally and within each aging
phenotype.

In [ ]:
questionnaire_source_columns = [
    column
    for column in baseline_header
    if column != baseline_id_column
]
questionnaire_schema_token = hashlib.sha256(
    (
        source_signature(baseline_archive_path)
        + "|"
        + baseline_member
        + "|"
        + "|".join(questionnaire_source_columns)
    ).encode("utf-8")
).hexdigest()[:12]
questionnaire_batch_root = (
    checkpoint_root
    / "questionnaire_all_baseline_fields"
    / f"schema_{questionnaire_schema_token}"
)
questionnaire_batch_root.mkdir(
    parents=True,
    exist_ok=True,
)
three_clock_participant_ids = (
    tri_age["participant_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)
questionnaire_cohort_token = hashlib.sha256(
    "|".join(sorted(three_clock_participant_ids)).encode("utf-8")
).hexdigest()[:12]
questionnaire_batch_root = (
    questionnaire_batch_root
    / f"cohort_{questionnaire_cohort_token}"
)
questionnaire_batch_root.mkdir(
    parents=True,
    exist_ok=True,
)
eligible_questionnaire_ids = set(three_clock_participant_ids)
questionnaire_batches = []
questionnaire_batch_manifest_rows = []
scanned_questionnaire_rows = 0
with zipfile.ZipFile(baseline_archive_path) as archive:
    with archive.open(baseline_member) as stream:
        raw_questionnaire_chunks = pd.read_csv(
            stream,
            dtype="string",
            chunksize=questionnaire_row_chunk_size,
            low_memory=False,
        )
        for batch_number, raw_chunk in enumerate(
            raw_questionnaire_chunks,
            1,
        ):
            start_row = scanned_questionnaire_rows
            scanned_questionnaire_rows += len(raw_chunk)
            stop_row = scanned_questionnaire_rows
            batch_path = (
                questionnaire_batch_root
                / f"rows_{start_row:09d}_{stop_row:09d}.parquet"
            )
            if (
                resume_completed_outputs
                and batch_path.is_file()
            ):
                batch_frame = pd.read_parquet(batch_path)
                status = "resumed"
            else:
                raw_chunk[baseline_id_column] = normalize_identifier(
                    raw_chunk[baseline_id_column]
                )
                batch_frame = raw_chunk[
                    raw_chunk[baseline_id_column].isin(
                        eligible_questionnaire_ids
                    )
                ].copy()
                batch_frame = batch_frame.rename(
                    columns={baseline_id_column: "participant_id"}
                )
                batch_frame.attrs = {}
                write_frame(batch_frame, batch_path)
                status = "written"
            questionnaire_batches.append(batch_frame)
            questionnaire_batch_manifest_rows.append(
                {
                    "batch_number": batch_number,
                    "start_row": start_row,
                    "stop_row": stop_row,
                    "source_rows": int(len(raw_chunk)),
                    "retained_participants": int(
                        batch_frame["participant_id"].nunique()
                    ),
                    "n_questionnaire_fields": int(
                        len(questionnaire_source_columns)
                    ),
                    "path": str(batch_path),
                    "status": status,
                }
            )
            print(
                f"[questionnaire {batch_number}] {status} raw rows "
                f"{start_row:,}:{stop_row:,}; retained "
                f"{batch_frame['participant_id'].nunique():,}",
                flush=True,
            )

matched_questionnaire_rows = (
    pd.concat(questionnaire_batches, ignore_index=True)
    if questionnaire_batches
    else pd.DataFrame(
        columns=["participant_id", *questionnaire_source_columns]
    )
)
if matched_questionnaire_rows["participant_id"].duplicated().any():
    matched_questionnaire_rows = (
        matched_questionnaire_rows.groupby(
            "participant_id",
            as_index=False,
        ).agg(
            {
                column: stable_value
                for column in questionnaire_source_columns
            }
        )
    )
all_questionnaire_answers = tri_age[["participant_id"]].merge(
    matched_questionnaire_rows,
    on="participant_id",
    how="left",
    validate="one_to_one",
)
if all_questionnaire_answers["participant_id"].duplicated().any():
    raise ValueError(
        "Matched all-questionnaire extract is not participant-unique"
    )
write_frame(
    all_questionnaire_answers,
    private_root
    / "all_baseline_questionnaire_answers_private.parquet",
)
questionnaire_batch_manifest = pd.DataFrame(
    questionnaire_batch_manifest_rows
)
write_frame(
    questionnaire_batch_manifest,
    statistics_root / "questionnaire_batch_manifest.csv",
)

questionnaire_missing_codes = {
    *numeric_missing_codes,
    "-77777",
    "-88889",
    "-99998",
    "NA",
    "N/A",
    "NAN",
    "NONE",
    "NULL",
    "MISSING",
    "DON'T KNOW",
    "DO NOT KNOW",
    "REFUSED",
}
questionnaire_reserved_columns = {
    "age_at_fundus_years",
    "sex_at_birth",
    "ethnicity_spirometry",
    "smoking_status",
    "bmi",
    "body_mass_index",
    "multimorbidity_selected_count",
    *COMORBIDITY_LABELS.keys(),
    *EPIGENETIC_SOURCE_VARIABLES.keys(),
    *EPIGENETIC_SOURCE_VARIABLES.values(),
}
questionnaire_identifier_pattern = re.compile(
    r"(?i)(^|_)(participant|entity|record|subject|person)_?id($|_)"
)
questionnaire_time_pattern = re.compile(
    r"(?i)(^|_)(date|time|timestamp|dob|birth_year|age)($|_)"
)

questionnaire_audit_rows = []
analyzable_questionnaire = {
    "participant_id": all_questionnaire_answers[
        "participant_id"
    ]
}
questionnaire_variable_types = {}
upper_missing_codes = {
    str(code).upper() for code in questionnaire_missing_codes
}
for variable in questionnaire_source_columns:
    raw = all_questionnaire_answers[variable]
    text_values = raw.astype("string").str.strip()
    missing_mask = (
        text_values.str.upper().isin(upper_missing_codes)
        | text_values.isna()
    )
    cleaned_text = text_values.mask(missing_mask)
    observed_count = int(cleaned_text.notna().sum())
    unique_count = int(cleaned_text.nunique(dropna=True))
    numeric_values = pd.to_numeric(
        cleaned_text,
        errors="coerce",
    )
    numeric_fraction = (
        float(numeric_values.notna().sum() / observed_count)
        if observed_count
        else 0.0
    )
    skip_reason = None
    variable_type = None
    if variable in questionnaire_reserved_columns:
        skip_reason = "already_analyzed_or_adjustment_variable"
    elif questionnaire_identifier_pattern.search(variable):
        skip_reason = "identifier"
    elif questionnaire_time_pattern.search(variable):
        skip_reason = "age_or_time_leakage"
    elif observed_count < minimum_questionnaire_observed:
        skip_reason = "insufficient_observed_participants"
    elif unique_count < 2:
        skip_reason = "constant"
    elif numeric_fraction >= 0.95 and unique_count > 10:
        variable_type = "numeric"
        analyzable_questionnaire[variable] = numeric_values
    elif unique_count <= maximum_questionnaire_categorical_levels:
        variable_type = "categorical"
        analyzable_questionnaire[variable] = cleaned_text
    else:
        skip_reason = "high_cardinality_or_free_text"
    questionnaire_audit_rows.append(
        {
            "variable": variable,
            "source_type": "raw_csv_string",
            "observed_participants": observed_count,
            "missing_fraction": float(
                1 - observed_count / len(all_questionnaire_answers)
            ),
            "unique_values": unique_count,
            "numeric_fraction": numeric_fraction,
            "analysis_type": variable_type,
            "analyzed": variable_type is not None,
            "skip_reason": skip_reason,
        }
    )
    if variable_type:
        questionnaire_variable_types[variable] = variable_type

questionnaire_clean = pd.DataFrame(
    analyzable_questionnaire
)
questionnaire_variable_audit = pd.DataFrame(
    questionnaire_audit_rows
)
write_frame(
    questionnaire_variable_audit,
    statistics_root / "questionnaire_variable_audit.csv",
)
write_frame(
    questionnaire_clean,
    private_root
    / "analyzable_questionnaire_answers_private.parquet",
)
print(
    f"Loaded {len(questionnaire_source_columns):,} scalar questionnaire "
    f"fields; {len(questionnaire_variable_types):,} passed analysis rules"
)
display(
    questionnaire_variable_audit.groupby(
        ["analyzed", "analysis_type", "skip_reason"],
        dropna=False,
    ).size().reset_index(name="variables")
)

### Questionnaire association models

Each retained answer is evaluated against retinal acceleration, mean
epigenetic acceleration, retina-specific discordance, and overall
three-clock dispersion using the same demographic and age adjustment
set as the comorbidity analyses.

In [ ]:
questionnaire_aging_outcomes = {
    "z_retinal_acceleration": "Retinal acceleration",
    "z_epigenetic_mean_acceleration": (
        "Mean epigenetic acceleration"
    ),
    "retina_specific_discordance": (
        "Retina-specific discordance"
    ),
    "z_three_clock_dispersion": (
        "Overall three-clock dispersion"
    ),
}
questionnaire_analysis_base_columns = [
    "participant_id",
    *questionnaire_aging_outcomes,
    "chronological_age",
    "chronological_age_sq",
    "log_n_embedded_images",
    *association_categorical_columns,
]
for continuous in ("bmi", "body_mass_index"):
    if continuous in association_covariate_terms:
        questionnaire_analysis_base_columns.append(continuous)
questionnaire_analysis_frame = tri_age[
    list(dict.fromkeys(questionnaire_analysis_base_columns))
].merge(
    questionnaire_clean,
    on="participant_id",
    how="left",
    validate="one_to_one",
)

questionnaire_test_rows = []
questionnaire_level_rows = []
questionnaire_failure_rows = []
for variable, variable_type in (
    questionnaire_variable_types.items()
):
    for outcome, outcome_label in (
        questionnaire_aging_outcomes.items()
    ):
        model_columns = [
            outcome,
            "chronological_age",
            "chronological_age_sq",
            "log_n_embedded_images",
            *association_categorical_columns,
            variable,
        ]
        for continuous in ("bmi", "body_mass_index"):
            if continuous in association_covariate_terms:
                model_columns.append(continuous)
        work = questionnaire_analysis_frame[
            list(dict.fromkeys(model_columns))
        ].copy()
        work[outcome] = pd.to_numeric(
            work[outcome], errors="coerce"
        )
        if variable_type == "numeric":
            work["question_value_numeric"] = safe_zscore(
                pd.to_numeric(work[variable], errors="coerce")
            )
            work = work.dropna(
                subset=[outcome, "question_value_numeric"]
            )
            question_term = "question_value_numeric"
        else:
            question_values = (
                work[variable].astype("string").str.strip()
            )
            level_counts = question_values.value_counts(
                dropna=True
            )
            frequent_levels = level_counts[
                level_counts >= minimum_questionnaire_level_n
            ].index
            question_values = question_values.where(
                question_values.isna()
                | question_values.isin(frequent_levels),
                "Other/low-frequency",
            )
            work["question_value_categorical"] = pd.Categorical(
                question_values.astype("object")
            )
            work = work.dropna(
                subset=[
                    outcome,
                    "question_value_categorical",
                ]
            )
            question_term = "C(question_value_categorical)"
            if work[
                "question_value_categorical"
            ].nunique() < 2:
                questionnaire_failure_rows.append(
                    {
                        "variable": variable,
                        "outcome": outcome,
                        "status": "skipped_fewer_than_two_levels",
                    }
                )
                continue
        if len(work) < minimum_questionnaire_observed:
            questionnaire_failure_rows.append(
                {
                    "variable": variable,
                    "outcome": outcome,
                    "status": "skipped_insufficient_complete_cases",
                    "n": int(len(work)),
                }
            )
            continue
        base_formula = (
            f"{outcome} ~ "
            + " + ".join(association_covariate_terms)
        )
        full_formula = base_formula + f" + {question_term}"
        try:
            base_fit = smf.ols(
                base_formula,
                data=work,
                missing="drop",
            ).fit()
            classical_full = smf.ols(
                full_formula,
                data=work,
                missing="drop",
            ).fit()
            robust_full = smf.ols(
                full_formula,
                data=work,
                missing="drop",
            ).fit(cov_type="HC3")
            nested_f, nested_p, df_difference = (
                classical_full.compare_f_test(base_fit)
            )
            result = {
                "variable": variable,
                "analysis_type": variable_type,
                "outcome": outcome,
                "outcome_label": outcome_label,
                "n": int(classical_full.nobs),
                "observed_levels": int(
                    (
                        work["question_value_categorical"].nunique()
                        if variable_type == "categorical"
                        else work[variable].nunique(dropna=True)
                    )
                ),
                "base_r2": float(base_fit.rsquared),
                "full_r2": float(classical_full.rsquared),
                "incremental_r2": float(
                    classical_full.rsquared - base_fit.rsquared
                ),
                "omnibus_f": float(nested_f),
                "p_value": float(nested_p),
                "df_difference": float(df_difference),
            }
            if variable_type == "numeric":
                confidence = robust_full.conf_int().loc[
                    "question_value_numeric"
                ]
                result.update(
                    {
                        "standardized_coefficient": float(
                            robust_full.params[
                                "question_value_numeric"
                            ]
                        ),
                        "coefficient_ci_low": float(
                            confidence.iloc[0]
                        ),
                        "coefficient_ci_high": float(
                            confidence.iloc[1]
                        ),
                        "robust_coefficient_p": float(
                            robust_full.pvalues[
                                "question_value_numeric"
                            ]
                        ),
                    }
                )
            else:
                confidence = robust_full.conf_int()
                prefix = "C(question_value_categorical)[T."
                for term in robust_full.params.index:
                    if not term.startswith(prefix):
                        continue
                    questionnaire_level_rows.append(
                        {
                            "variable": variable,
                            "outcome": outcome,
                            "outcome_label": outcome_label,
                            "term": term,
                            "comparison_level": term[
                                len(prefix):-1
                            ],
                            "n": int(robust_full.nobs),
                            "coefficient": float(
                                robust_full.params[term]
                            ),
                            "ci_low": float(
                                confidence.loc[term, 0]
                            ),
                            "ci_high": float(
                                confidence.loc[term, 1]
                            ),
                            "p_value": float(
                                robust_full.pvalues[term]
                            ),
                        }
                    )
            questionnaire_test_rows.append(result)
        except Exception as error:
            questionnaire_failure_rows.append(
                {
                    "variable": variable,
                    "analysis_type": variable_type,
                    "outcome": outcome,
                    "status": f"failed:{type(error).__name__}",
                    "error_message": str(error)[:500],
                    "n": int(len(work)),
                }
            )

questionnaire_tests = pd.DataFrame(questionnaire_test_rows)
questionnaire_level_coefficients = pd.DataFrame(
    questionnaire_level_rows
)
questionnaire_failures = pd.DataFrame(
    questionnaire_failure_rows
)
if not questionnaire_tests.empty:
    questionnaire_tests["fdr_q_global"] = (
        benjamini_hochberg(
            questionnaire_tests["p_value"].to_numpy(float)
        )
    )
    questionnaire_tests["fdr_q_within_outcome"] = np.nan
    for outcome, index in questionnaire_tests.groupby(
        "outcome"
    ).groups.items():
        questionnaire_tests.loc[
            index,
            "fdr_q_within_outcome",
        ] = benjamini_hochberg(
            questionnaire_tests.loc[
                index,
                "p_value",
            ].to_numpy(float)
        )
    questionnaire_tests["significant_global_fdr_0_05"] = (
        questionnaire_tests["fdr_q_global"] < 0.05
    )
if not questionnaire_level_coefficients.empty:
    questionnaire_level_coefficients["fdr_q_global"] = (
        benjamini_hochberg(
            questionnaire_level_coefficients[
                "p_value"
            ].to_numpy(float)
        )
    )
write_frame(
    questionnaire_tests,
    statistics_root / "questionnaire_phenome_scan_tests.csv",
)
write_frame(
    questionnaire_level_coefficients,
    statistics_root
    / "questionnaire_categorical_level_coefficients.csv",
)
write_frame(
    questionnaire_failures,
    statistics_root / "questionnaire_phenome_scan_failures.csv",
)
display(
    questionnaire_tests.sort_values(
        ["fdr_q_global", "p_value"]
    ).head(100).round(6)
    if not questionnaire_tests.empty
    else questionnaire_tests
)

if not questionnaire_tests.empty:
    questionnaire_plot = questionnaire_tests.copy()
    questionnaire_plot["minus_log10_q"] = -np.log10(
        questionnaire_plot["fdr_q_global"].clip(lower=1e-300)
    )
    figure, axes = plt.subplots(
        2,
        2,
        figsize=(16, 13),
        layout="constrained",
    )
    for axis, (outcome, outcome_label) in zip(
        axes.ravel(),
        questionnaire_aging_outcomes.items(),
    ):
        plot = (
            questionnaire_plot[
                questionnaire_plot["outcome"].eq(outcome)
            ]
            .nsmallest(15, "fdr_q_global")
            .sort_values("minus_log10_q")
        )
        positions = np.arange(len(plot))
        colors = np.where(
            plot["analysis_type"].eq("numeric"),
            "#4C72B0",
            "#DD8452",
        )
        axis.barh(
            positions,
            plot["minus_log10_q"],
            color=colors,
        )
        labels = [
            value
            if len(value) <= 48
            else value[:45] + "..."
            for value in plot["variable"].astype(str)
        ]
        axis.set_yticks(positions, labels)
        axis.axvline(
            -np.log10(0.05),
            color="black",
            linestyle="--",
        )
        axis.set(
            title=outcome_label,
            xlabel="−log10(global FDR q-value)",
            ylabel="",
        )
    figure.suptitle(
        "Questionnaire-wide associations with aging phenotypes",
        y=1.02,
    )
    figure.savefig(
        figure_root
        / "figure_10_questionnaire_phenome_scan.png",
        dpi=220,
        bbox_inches="tight",
        pad_inches=0.3,
    )
    plt.show()
    plt.close(figure)

## 15. Longitudinal retinal-aging trajectory

Baseline and F1 out-of-fold retinal predictions are calibrated jointly
with participant-grouped folds. Baseline epigenetic acceleration is then
tested against follow-up retinal acceleration while controlling for the
participant's baseline retinal acceleration. Because methylation is
baseline-only, this is a prospective association—not a comparison of
changes in both clocks.

In [ ]:
retinal_visit = chronological_oof[
    [
        "participant_id",
        "visit",
        "chronological_age",
        "retinal_age_oof",
    ]
].copy()
(
    retinal_visit["retinal_acceleration_cf_visit"],
    retinal_visit["retinal_expected_cf_visit"],
) = cross_fitted_spline_residual(
    retinal_visit,
    "retinal_age_oof",
    "chronological_age",
    "participant_id",
)
baseline_retinal = retinal_visit[
    retinal_visit["visit"].eq("BL")
].rename(
    columns={
        "chronological_age": "chronological_age_bl",
        "retinal_age_oof": "retinal_age_bl",
        "retinal_acceleration_cf_visit": (
            "retinal_acceleration_bl"
        ),
    }
)
followup_retinal = retinal_visit[
    retinal_visit["visit"].eq("F1")
].rename(
    columns={
        "chronological_age": "chronological_age_f1",
        "retinal_age_oof": "retinal_age_f1",
        "retinal_acceleration_cf_visit": (
            "retinal_acceleration_f1"
        ),
    }
)
longitudinal = baseline_retinal[
    [
        "participant_id",
        "chronological_age_bl",
        "retinal_age_bl",
        "retinal_acceleration_bl",
    ]
].merge(
    followup_retinal[
        [
            "participant_id",
            "chronological_age_f1",
            "retinal_age_f1",
            "retinal_acceleration_f1",
        ]
    ],
    on="participant_id",
    how="inner",
    validate="one_to_one",
)
longitudinal["followup_years"] = (
    longitudinal["chronological_age_f1"]
    - longitudinal["chronological_age_bl"]
)
longitudinal = longitudinal[
    longitudinal["followup_years"].between(0.25, 15)
].copy()
longitudinal["retinal_acceleration_change"] = (
    longitudinal["retinal_acceleration_f1"]
    - longitudinal["retinal_acceleration_bl"]
)
longitudinal["retinal_acceleration_change_per_year"] = (
    longitudinal["retinal_acceleration_change"]
    / longitudinal["followup_years"]
)
phenotype_columns = [
    "participant_id",
    "z_retinal_acceleration",
    "z_horvath_acceleration",
    "z_hannum_acceleration",
    "z_epigenetic_mean_acceleration",
    "shared_aging_pc",
    "retina_vs_epigenetic_pc",
    *association_categorical_columns,
]
longitudinal = longitudinal.merge(
    tri_age[phenotype_columns],
    on="participant_id",
    how="inner",
    validate="one_to_one",
)

longitudinal_model_definitions = {
    "mean_epigenetic_acceleration": [
        "z_epigenetic_mean_acceleration"
    ],
    "separate_epigenetic_clocks": [
        "z_horvath_acceleration",
        "z_hannum_acceleration",
    ],
    "orthogonal_shared_and_discordant": [
        "shared_aging_pc",
        "retina_vs_epigenetic_pc",
    ],
}
longitudinal_rows = []
longitudinal_failures = []
longitudinal_covariates = [
    "retinal_acceleration_bl",
    "chronological_age_bl",
    "followup_years",
    *[
        f"C({column})"
        for column in association_categorical_columns
    ],
]
for model_name, predictors in (
    longitudinal_model_definitions.items()
):
    formula = (
        "retinal_acceleration_f1 ~ "
        + " + ".join([*longitudinal_covariates, *predictors])
    )
    try:
        fit = smf.ols(
            formula,
            data=longitudinal,
            missing="drop",
        ).fit(cov_type="HC3")
        confidence = fit.conf_int()
        for predictor in predictors:
            longitudinal_rows.append(
                {
                    "model": model_name,
                    "predictor": predictor,
                    "n": int(fit.nobs),
                    "coefficient": float(fit.params[predictor]),
                    "ci_low": float(confidence.loc[predictor, 0]),
                    "ci_high": float(confidence.loc[predictor, 1]),
                    "p_value": float(fit.pvalues[predictor]),
                    "outcome": "follow-up retinal acceleration",
                }
            )
    except Exception as error:
        longitudinal_failures.append(
            {
                "model": model_name,
                "status": f"failed:{type(error).__name__}",
                "error_message": str(error)[:500],
            }
        )
longitudinal_associations = pd.DataFrame(longitudinal_rows)
if not longitudinal_associations.empty:
    longitudinal_associations["fdr_q_value"] = (
        benjamini_hochberg(
            longitudinal_associations[
                "p_value"
            ].to_numpy(float)
        )
    )
write_frame(
    longitudinal,
    private_root / "longitudinal_retinal_aging_private.parquet",
)
write_frame(
    longitudinal_associations,
    statistics_root
    / "baseline_epigenetic_to_followup_retinal_associations.csv",
)
write_frame(
    pd.DataFrame(longitudinal_failures),
    statistics_root / "longitudinal_model_failures.csv",
)
display(longitudinal_associations.round(5))

if not longitudinal.empty:
    figure, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5.8),
        layout="constrained",
    )
    color = longitudinal["z_epigenetic_mean_acceleration"]
    scatter = axes[0].scatter(
        longitudinal["retinal_acceleration_bl"],
        longitudinal["retinal_acceleration_f1"],
        c=color,
        cmap="coolwarm",
        s=22,
        alpha=0.65,
    )
    limits = [
        float(
            longitudinal[
                [
                    "retinal_acceleration_bl",
                    "retinal_acceleration_f1",
                ]
            ].min().min()
        ),
        float(
            longitudinal[
                [
                    "retinal_acceleration_bl",
                    "retinal_acceleration_f1",
                ]
            ].max().max()
        ),
    ]
    axes[0].plot(limits, limits, "--", color="black")
    axes[0].set(
        title="Retinal acceleration stability",
        xlabel="Baseline retinal acceleration (years)",
        ylabel="Follow-up retinal acceleration (years)",
    )
    figure.colorbar(
        scatter,
        ax=axes[0],
        label="Baseline mean epigenetic acceleration (SD)",
    )
    sns.regplot(
        data=longitudinal,
        x="z_epigenetic_mean_acceleration",
        y="retinal_acceleration_change_per_year",
        scatter_kws={"s": 18, "alpha": 0.45},
        line_kws={"color": "black"},
        ax=axes[1],
    )
    axes[1].axhline(0, color="gray", linestyle="--")
    axes[1].set(
        title="Baseline epigenetic age and retinal-aging rate",
        xlabel="Baseline mean epigenetic acceleration (SD)",
        ylabel="Annual change in retinal acceleration (years/year)",
    )
    figure.suptitle(
        "Longitudinal retinal aging among participants with methylation",
        y=1.02,
    )
    figure.savefig(
        figure_root / "figure_11_longitudinal_retinal_aging.png",
        dpi=220,
        bbox_inches="tight",
        pad_inches=0.25,
    )
    plt.show()
    plt.close(figure)

reliability_capability = {
    "eye_level_reliability_available": False,
    "reason": (
        "The configured RETFound input is already aggregated to one "
        "embedding per participant-visit. Left/right eye predictions "
        "must be computed from the image-level embedding batches."
    ),
    "participant_visit_longitudinal_analysis_available": bool(
        len(longitudinal) > 0
    ),
    "n_longitudinal_three_clock_participants": int(
        longitudinal["participant_id"].nunique()
    ),
}
write_json(
    reliability_capability,
    statistics_root / "reliability_capability.json",
)
print(reliability_capability)

## 16. Final run summary

The generated README provides exact cohort counts, primary metrics,
significant demographic contributions, paired-test conclusions, paths,
and interpretation limitations for manuscript development.

In [ ]:
significant_demographics = (
    demographic_models[
        demographic_models.get(
            "significant_fdr_0_05",
            pd.Series(False, index=demographic_models.index),
        ).fillna(False)
    ][
        [
            "demographic_label",
            "outcome_label",
            "n",
            "incremental_r2",
            "fdr_q_value",
        ]
    ].to_dict("records")
    if not demographic_models.empty
    else []
)
supported_distance_hypotheses = distance_tests[
    distance_tests["supports_closer_after_fdr"]
].to_dict("records")
age_stratified_statistical_findings = (
    age_stratified_distance_tests[
        age_stratified_distance_tests[
            "supports_closer_statistically"
        ]
    ].to_dict("records")
    if not age_stratified_distance_tests.empty
    else []
)
calibration_sensitive_tail_findings = (
    age_stratified_distance_tests[
        age_stratified_distance_tests[
            "supports_closer_statistically"
        ]
        & age_stratified_distance_tests[
            "tail_calibration_warning"
        ]
    ].to_dict("records")
    if not age_stratified_distance_tests.empty
    else []
)
publication_supported_age_stratified_findings = (
    age_stratified_distance_tests[
        age_stratified_distance_tests[
            "publication_support_flag"
        ]
    ].to_dict("records")
    if not age_stratified_distance_tests.empty
    else []
)
significant_comorbidity_clock_associations = (
    comorbidity_associations[
        comorbidity_associations.get(
            "significant_fdr_0_05",
            pd.Series(
                False,
                index=comorbidity_associations.index,
            ),
        ).fillna(False)
    ].to_dict("records")
    if not comorbidity_associations.empty
    else []
)
significant_multimorbidity_associations = (
    multimorbidity_associations[
        multimorbidity_associations.get(
            "fdr_q_value",
            pd.Series(
                np.nan,
                index=multimorbidity_associations.index,
            ),
        ).lt(0.05)
    ].to_dict("records")
    if not multimorbidity_associations.empty
    else []
)
significant_longitudinal_associations = (
    longitudinal_associations[
        longitudinal_associations.get(
            "fdr_q_value",
            pd.Series(
                np.nan,
                index=longitudinal_associations.index,
            ),
        ).lt(0.05)
    ].to_dict("records")
    if not longitudinal_associations.empty
    else []
)
best_predictive_increments = (
    predictive_metrics[
        predictive_metrics.get(
            "delta_auc_vs_base",
            pd.Series(
                np.nan,
                index=predictive_metrics.index,
            ),
        ).notna()
    ]
    .sort_values("delta_auc_vs_base", ascending=False)
    .groupby("outcome", as_index=False)
    .head(1)
    .to_dict("records")
    if not predictive_metrics.empty
    else []
)
significant_questionnaire_associations = (
    questionnaire_tests[
        questionnaire_tests.get(
            "significant_global_fdr_0_05",
            pd.Series(False, index=questionnaire_tests.index),
        ).fillna(False)
    ]
    .sort_values(["fdr_q_global", "p_value"])
    .head(50)
    .to_dict("records")
    if not questionnaire_tests.empty
    else []
)
questionnaire_analysis_counts = (
    questionnaire_variable_audit[
        questionnaire_variable_audit["analyzed"].fillna(False)
    ]
    .groupby("analysis_type")
    .size()
    .astype(int)
    .to_dict()
)
any_epigenetic_n = int(
    epigenetic["any_epigenetic_measure"].sum()
)
chronological_participant_metric = model_metrics[
    model_metrics["analysis"].eq(
        "Chronological age head: participant"
    )
].iloc[0]
readme = f'''# CLSA retinal and epigenetic aging run

## Cohort and inputs

- Quality-passing images represented by the RETFound rollup:
  {int(participant_visit["n_embedded_images"].sum()):,}
- Participant-visits with embeddings: {len(participant_visit):,}
- Unique participants with embeddings:
  {participant_visit["participant_id"].nunique():,}
- Participants with at least one released epigenetic phenotype:
  {any_epigenetic_n:,}
- Baseline participants with OOF retinal age and any epigenetic phenotype:
  {len(master):,}

## Chronological-age head

- Participant-level OOF MAE:
  {chronological_participant_metric["mae"]:.3f} years
- Participant-level OOF RMSE:
  {chronological_participant_metric["rmse"]:.3f} years
- Participant-level OOF R-squared:
  {chronological_participant_metric["r2"]:.4f}
- Pearson correlation:
  {chronological_participant_metric["pearson_r"]:.4f}
- Calibration slope:
  {chronological_participant_metric["calibration_slope"]:.4f}
- Central-age range (10th to 90th percentile):
  {age_q10:.2f} to {age_q90:.2f} years

## Primary paired-distance conclusion

The primary comparison uses retinal age from the chronological-age head.
Supported hypotheses require a negative bootstrap confidence interval and
both one-sided Wilcoxon and sign-flip FDR q-values below 0.05.

Supported hypotheses: {json.dumps(supported_distance_hypotheses, default=str)}

## Age-tail sensitivity

Tail analyses are exploratory. A statistically positive tail result is
marked calibration-sensitive when the chronological RETFound model has
an absolute mean error of at least two years in that age stratum. Such
results are retained in the audit table but are excluded from
manuscript-support flags because regression to the cohort mean can make
two underestimated ages appear artificially close.

Statistically positive age-stratified findings:
{json.dumps(age_stratified_statistical_findings, default=str)}

Calibration-sensitive tail findings:
{json.dumps(calibration_sensitive_tail_findings, default=str)}

Age-stratified findings suitable for manuscript support:
{json.dumps(publication_supported_age_stratified_findings, default=str)}

## Demographic explanatory contribution

Demographic factors are evaluated separately in adjusted nested models.
Incremental R-squared is variance added beyond age, available
cardiometabolic comorbidities, and complementary demographic covariates.

FDR-significant demographic models:
{json.dumps(significant_demographics, default=str)}

## Patient-level three-clock phenotypes

Acceleration residuals were estimated with participant-grouped,
cross-fitted chronological-age splines before standardization. The
primary continuous phenotypes separate shared aging, retina-specific
discordance, epigenetic-specific discordance, and overall three-clock
dispersion. Percentile-based profiles are descriptive only.

Profile summary:
{json.dumps(profile_summary.to_dict("records"), default=str)}

## Comorbidity associations

Binary comorbidities were modeled as outcomes using mutually adjusted
retinal and mean epigenetic acceleration and, separately, orthogonal
three-clock components. Reported associations are cross-sectional.

FDR-significant clock-comorbidity associations:
{json.dumps(significant_comorbidity_clock_associations, default=str)}

FDR-significant multimorbidity associations:
{json.dumps(significant_multimorbidity_associations, default=str)}

Best out-of-fold disease-prediction increments:
{json.dumps(best_predictive_increments, default=str)}

## Questionnaire-wide phenome scan

All raw baseline CoPv7_Qx_CANUE_PA_BS questionnaire fields were loaded
for the three-clock cohort in resumable row batches. Identifiers,
age/date leakage, prior adjustment variables, and fields failing the
prespecified completeness/cardinality rules were audited but not tested.
This is an exploratory association scan; global FDR is the primary
multiplicity correction.

Raw questionnaire fields loaded:
{len(questionnaire_source_columns):,}

Analyzable fields by type:
{json.dumps(questionnaire_analysis_counts, default=str)}

Total questionnaire-aging tests:
{len(questionnaire_tests):,}

Globally FDR-significant questionnaire-aging associations (first 50):
{json.dumps(significant_questionnaire_associations, default=str)}

Model failures/skips recorded:
{len(questionnaire_failures):,}

## Longitudinal retinal aging

Baseline epigenetic acceleration was tested against F1 retinal
acceleration while controlling for baseline retinal acceleration,
chronological age, follow-up duration, and available demographics.

Participants with baseline and F1 retinal predictions plus epigenetics:
{longitudinal["participant_id"].nunique():,}

FDR-significant longitudinal associations:
{json.dumps(significant_longitudinal_associations, default=str)}

Eye-level reliability was not estimated because the configured input has
already been aggregated across images within participant-visit.

## Interpretation

Retinal and epigenetic ages are related biomarkers, not interchangeable
measurements. A smaller retinal-epigenetic distance does not prove a shared
causal aging mechanism. The epigenetic-trained retinal heads are secondary
prediction analyses and must not be used as the primary evidence of
retinal-epigenetic convergence. Released racial/cultural background is
self-report and is not genetic ancestry. Spirometry ethnicity is a
separate operational variable and should not be substituted for race.
Cross-sectional differences can reflect cohort composition, health,
imaging, selection, and residual confounding.
'''
readme = "\n".join(
    line.strip() for line in readme.splitlines()
).strip() + "\n"
(output_root / "RUN_README.md").write_text(
    readme,
    encoding="utf-8",
)
write_json(
    {
        "completed": True,
        "output_root": str(output_root),
        "participant_embedding_path": str(participant_embedding_path),
        "baseline_member": baseline_member,
        "n_images_represented": int(
            participant_visit["n_embedded_images"].sum()
        ),
        "n_embedding_participants": int(
            participant_visit["participant_id"].nunique()
        ),
        "n_any_epigenetic": any_epigenetic_n,
        "n_baseline_three_age": int(len(master)),
        "absolute_clocks": ABSOLUTE_CLOCKS,
        "acceleration_measures": ACCELERATION_MEASURES,
        "primary_retinal_age_model_target": "chronological_age",
        "participant_grouped_oof": True,
        "supported_distance_hypotheses": supported_distance_hypotheses,
        "age_stratified_statistical_findings": (
            age_stratified_statistical_findings
        ),
        "calibration_sensitive_tail_findings": (
            calibration_sensitive_tail_findings
        ),
        "publication_supported_age_stratified_findings": (
            publication_supported_age_stratified_findings
        ),
        "significant_demographic_models": significant_demographics,
        "three_clock_profile_summary": profile_summary.to_dict(
            "records"
        ),
        "significant_comorbidity_clock_associations": (
            significant_comorbidity_clock_associations
        ),
        "significant_multimorbidity_associations": (
            significant_multimorbidity_associations
        ),
        "best_predictive_increments": best_predictive_increments,
        "n_raw_questionnaire_fields": int(
            len(questionnaire_source_columns)
        ),
        "questionnaire_analysis_counts": (
            questionnaire_analysis_counts
        ),
        "n_questionnaire_aging_tests": int(
            len(questionnaire_tests)
        ),
        "significant_questionnaire_associations": (
            significant_questionnaire_associations
        ),
        "n_questionnaire_model_failures_or_skips": int(
            len(questionnaire_failures)
        ),
        "n_longitudinal_three_clock_participants": int(
            longitudinal["participant_id"].nunique()
        ),
        "significant_longitudinal_associations": (
            significant_longitudinal_associations
        ),
    },
    output_root / "run_manifest.json",
)
print(readme)
print("Epigenetics notebook complete:", output_root)